# AGORA — Data Processing Pipeline (`raw_train_set`, `raw_val_set`, `raw_test_set`)

Rigorous MELD extraction and audio standardization based on `ser_sft.ipynb` and `ser_grpo_v2.ipynb`:
- Loads official MELD splits (`train`, `dev` -> `val`, `test`).
- Standardizes all audio files in parallel to **16 kHz, mono (16-bit PCM WAV)** inside `DATA_DIR`.
- Applies **only basic preprocessing** (no laugh removal yet) to `raw_train_set`, `raw_val_set`, and `raw_test_set`.
- Saves standardized `raw_train_set.csv`, `raw_val_set.csv`, and `raw_test_set.csv` to `DATA_DIR`.


In [1]:
!pip install -q whisperx
!pip install -q opensmile
!pip install -q cmudict g2p_en

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from math import gcd
import os
import shutil
import subprocess
import time
from typing import Dict, Optional, Tuple

import numpy as np
import pandas as pd
import soundfile as sf
from tqdm.auto import tqdm


## 1. Paths & Configuration


In [3]:
def mount_drive():
    """Mount Google Drive in Colab."""
    if Path("/content").exists():
        from google.colab import drive
        drive.mount("/content/drive")

mount_drive()

Mounted at /content/drive


In [4]:
IS_COLAB = Path("/content/drive/MyDrive").exists()

if IS_COLAB:
    MELD_DIR = Path("/content/prep_meld")
    DRIVE_ARCHIVE = Path("/content/drive/MyDrive/SER_DPO/MELD/MELD.tar")
    DRIVE_DATA_DIR: Optional[Path] = Path("/content/drive/MyDrive/AGORA_SER/data/meld_16k_mono")
    DATA_DIR = Path("/content/agora_work/meld_16k_mono")  # Fast local SSD workspace
else:
    MELD_DIR = Path("/home/fuganti/Projects/AFTA/data/prep_meld")
    DRIVE_ARCHIVE = Path("/content/drive/MyDrive/SER_DPO/MELD/MELD.tar")
    DRIVE_DATA_DIR = None
    DATA_DIR = Path("/home/fuganti/Projects/OPERA/data/meld_16k_mono")

TARGET_SR = 16000
TARGET_CHANNELS = 1
MAX_WORKERS = min(8, os.cpu_count() or 4)
OVERWRITE_AUDIO = False

# Mapping from canonical split name -> (MELD directory name, MELD CSV prefix)
SPLIT_CONFIG: Dict[str, Tuple[str, str]] = {
    "train": ("preprocessed_train_splits", "train"),
    "val": ("preprocessed_dev_splits_complete", "dev"),
    "test": ("preprocessed_output_repeated_splits_test", "test"),
}


def all_clean_outputs_cached(data_dir: Path = DATA_DIR) -> bool:
    """Return True when EMO-TAG, eGeMAPSv02, and quartile artifacts already exist locally."""
    clean_dir = Path(data_dir) / "emo_tag_clean"
    required = (
        "clean_train_set_with_cues.csv",
        "clean_val_set_with_cues.csv",
        "train_egemaps.csv",
        "val_egemaps.csv",
        "gender_acoustic_quartiles.json",
    )
    return all((clean_dir / name).is_file() for name in required)


DATA_DIR.mkdir(parents=True, exist_ok=True)
if DRIVE_DATA_DIR is not None:
    DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)
    # One-time restore if a packed tar or EMO-TAG checkpoints already exist on Drive
    drive_tar = DRIVE_DATA_DIR / "meld_16k_mono_clean.tar"
    restored_from_tar = False
    if drive_tar.exists() and not (DATA_DIR / "emo_tag_clean").exists():
        print(f"Restoring saved clean bundle from {drive_tar} -> {DATA_DIR}...")
        subprocess.run(["tar", "-xf", str(drive_tar), "-C", str(DATA_DIR.parent)], check=True)
        restored_from_tar = True
    elif (DRIVE_DATA_DIR / "emo_tag_clean").exists() and not (DATA_DIR / "emo_tag_clean").exists():
        print("Syncing existing emo_tag_clean checkpoints from Drive to local SSD...")
        subprocess.run(
            ["rsync", "-a", str(DRIVE_DATA_DIR / "emo_tag_clean") + "/", str(DATA_DIR / "emo_tag_clean") + "/"],
            check=True,
        )

    # Prefer separately saved Drive manifests over tar contents (immune to Drive FUSE mtime skew)
    drive_clean_dir = DRIVE_DATA_DIR / "emo_tag_clean"
    local_clean_dir = DATA_DIR / "emo_tag_clean"
    if drive_clean_dir.exists() and local_clean_dir.exists():
        for ext in ("*.csv", "*.json"):
            for src_meta in drive_clean_dir.glob(ext):
                dst_meta = local_clean_dir / src_meta.name
                if restored_from_tar or not dst_meta.exists():
                    shutil.copyfile(src_meta, dst_meta)
    for csv_p in local_clean_dir.glob("*.csv") if local_clean_dir.exists() else []:
        txt = csv_p.read_text(encoding="utf-8")
        if str(DRIVE_DATA_DIR) in txt:
            csv_p.write_text(txt.replace(str(DRIVE_DATA_DIR), str(DATA_DIR)), encoding="utf-8")

print(f"IS_COLAB        : {IS_COLAB}")
print(f"MELD_DIR        : {MELD_DIR} (exists={MELD_DIR.exists()})")
print(f"LOCAL WORKSPACE : {DATA_DIR}")
print(f"DRIVE BACKUP    : {DRIVE_DATA_DIR}")
print(f"CLEAN CACHED    : {all_clean_outputs_cached(DATA_DIR)}")
print(f"Workers         : {MAX_WORKERS}")

Restoring saved clean bundle from /content/drive/MyDrive/AGORA_SER/data/meld_16k_mono/meld_16k_mono_clean.tar -> /content/agora_work/meld_16k_mono...
IS_COLAB        : True
MELD_DIR        : /content/prep_meld (exists=False)
LOCAL WORKSPACE : /content/agora_work/meld_16k_mono
DRIVE BACKUP    : /content/drive/MyDrive/AGORA_SER/data/meld_16k_mono
CLEAN CACHED    : True
Workers         : 8


## 2. MELD Extraction & Metadata Helpers
Consistent with `load_meld_split` / `load_meld` in `ser_sft.ipynb` and `ser_grpo_v2.ipynb`.


In [5]:
def extract_tar(archive_path: Path, destination: Path) -> None:
    """Extract a .tar archive to destination."""
    archive_path = Path(archive_path)
    destination = Path(destination)

    if not archive_path.exists():
        raise FileNotFoundError(f"Archive not found: {archive_path}")

    destination.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {archive_path.name} -> {destination}")
    start = time.time()

    subprocess.run(
        ["tar", "-xf", str(archive_path), "-C", str(destination)],
        check=True,
    )

    elapsed = time.time() - start
    print(f"Extraction completed in {elapsed:.1f}s")


def dataset_info(path: Path) -> None:
    """Print basic information about an extracted dataset."""
    path = Path(path)
    n_files = sum(1 for p in path.rglob("*") if p.is_file())
    size = subprocess.run(
        ["du", "-sh", str(path)],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.split()[0]
    print(f"Directory: {path} | Files: {n_files:,} | Size: {size}")


def find_file(root: Path, pattern: str) -> Path:
    """Find the first file matching a pattern."""
    matches = list(Path(root).rglob(pattern))
    if not matches:
        raise FileNotFoundError(f"Could not find '{pattern}' under {root}")
    return matches[0]


def resolve_split_dir(destination: Path, split: str) -> Tuple[Path, str]:
    """Resolve split directory and MELD CSV prefix for train/val(dev)/test."""
    destination = Path(destination)
    if split in SPLIT_CONFIG:
        folder_name, csv_prefix = SPLIT_CONFIG[split]
        candidate = destination / folder_name
        if candidate.exists():
            return candidate, csv_prefix

    # Fallback patterns (e.g., preprocessed_dev_splits, dev_splits_complete, etc.)
    prefix = "dev" if split in ("val", "dev") else split
    candidates = [
        destination / f"preprocessed_{prefix}_splits",
        destination / f"preprocessed_{prefix}_splits_complete",
        destination / f"preprocessed_output_repeated_splits_{prefix}",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate, prefix

    raise FileNotFoundError(f"Split directory for '{split}' not found under {destination}")


In [6]:
def load_meld_split(split_dir: Path, split: str = "train") -> pd.DataFrame:
    """
    Load a MELD split and resolve audio paths.

    Expected files in split_dir:
        {split}_preprocessed.csv
        {split}_sent_emo.csv
        *.wav
    """
    split_dir = Path(split_dir)
    audio_files = list(split_dir.rglob("*.wav"))
    preprocessed_csv = find_file(split_dir, f"{split}_preprocessed.csv")
    emotion_csv = find_file(split_dir, f"{split}_sent_emo.csv")

    df = pd.read_csv(preprocessed_csv)
    df_emo = pd.read_csv(emotion_csv)

    # Resolve audio filenames -> absolute paths
    audio_map = {path.name: str(path.resolve()) for path in audio_files}
    df["audio_path"] = (
        df["audio_path"]
        .astype(str)
        .map(lambda x: Path(x).name)
        .map(audio_map)
    )

    # Normalize merge keys
    df["dialogue"] = df["dialogue"].astype(int)
    df["utterance"] = df["utterance"].astype(int)
    df_emo["Dialogue_ID"] = df_emo["Dialogue_ID"].astype(int)
    df_emo["Utterance_ID"] = df_emo["Utterance_ID"].astype(int)

    # Merge MELD metadata
    df = df.merge(
        df_emo[["Dialogue_ID", "Utterance_ID", "Emotion", "Utterance", "Speaker"]],
        left_on=["dialogue", "utterance"],
        right_on=["Dialogue_ID", "Utterance_ID"],
        how="left",
    )

    df = (
        df.drop(columns=["Dialogue_ID", "Utterance_ID"])
        .rename(
            columns={
                "Utterance": "text",
                "Emotion": "emotion",
                "Speaker": "speaker",
            }
        )
    )

    if "time" in df.columns and "duration" not in df.columns:
        df["duration"] = pd.to_numeric(df["time"], errors="coerce")

    missing_audio = int(df["audio_path"].isna().sum())
    missing_emotion = int(df["emotion"].isna().sum())

    print(f"[{split}] Loaded {len(df):,} samples")
    print(f"[{split}] Audio matched   : {len(df) - missing_audio:,}/{len(df):,}")
    if missing_audio:
        print(f"[{split}] Warning: {missing_audio:,} audio files missing")
    print(f"[{split}] Metadata matched: {len(df) - missing_emotion:,}/{len(df):,}")
    if missing_emotion:
        print(f"[{split}] Warning: {missing_emotion:,} metadata rows missing")

    return df


def load_meld(
    archive_path: Path = DRIVE_ARCHIVE,
    destination: Path = MELD_DIR,
    split: str = "train",
) -> pd.DataFrame:
    """Load (and optionally extract) a MELD split."""
    destination = Path(destination)
    if not destination.exists():
        extract_tar(archive_path, destination)

    split_dir, csv_prefix = resolve_split_dir(destination, split)
    return load_meld_split(split_dir, csv_prefix)


## 3. Load Raw Splits (`raw_train_set`, `raw_val_set`, `raw_test_set`)


In [7]:
#dataset_info(MELD_DIR)

_raw_csvs = {s: DATA_DIR / f"raw_{s}_set.csv" for s in ("train", "val", "test")}
if not OVERWRITE_AUDIO and all(p.is_file() for p in _raw_csvs.values()):
    print(f"[Load MELD] Skipping MELD.tar extraction — loading cached raw CSVs from {DATA_DIR}")
    raw_train_set = pd.read_csv(_raw_csvs["train"])
    raw_val_set = pd.read_csv(_raw_csvs["val"])
    raw_test_set = pd.read_csv(_raw_csvs["test"])
elif not OVERWRITE_AUDIO and all_clean_outputs_cached(DATA_DIR):
    print(f"[Load MELD] Skipping MELD.tar extraction — cleaned pipeline outputs are already cached in {DATA_DIR / 'emo_tag_clean'}")
    raw_train_set = None
    raw_val_set = None
    raw_test_set = None
else:
    raw_train_set = load_meld(archive_path=DRIVE_ARCHIVE, destination=MELD_DIR, split="train")
    raw_val_set = load_meld(archive_path=DRIVE_ARCHIVE, destination=MELD_DIR, split="val")
    raw_test_set = load_meld(archive_path=DRIVE_ARCHIVE, destination=MELD_DIR, split="test")

[Load MELD] Skipping MELD.tar extraction — loading cached raw CSVs from /content/agora_work/meld_16k_mono


## 3.1 Speaker Gender Classification (`F` / `M`) & Training Set Filtering
Classifies speakers into Female (`"F"`) and Male (`"M"`) using `KNOWN_SPEAKERS_GENDER` (Layer 1) and explicit gender keywords (`FEMALE_KEYWORDS`, `MALE_KEYWORDS`, Layer 2). Utterances whose speaker is not resolved by these rules (`gender is None`) are removed from the training set (`raw_train_set` / `clean_train_set`).


In [8]:
KNOWN_SPEAKERS_GENDER = {
    # Protagonistas (~83% do MELD)
    "Rachel": "F", "Monica": "F", "Phoebe": "F",
    "Joey": "M", "Chandler": "M", "Ross": "M",

    # Secundários recorrentes (Femininos)
    "Janice": "F", "Carol": "F", "Susan": "F", "Judy": "F", "Emily": "F",
    "Ursula": "F", "Mona": "F", "Charlie": "F", "Amy": "F", "Jill": "F",
    "Mindy": "F", "Estelle": "F", "Kathy": "F", "Elizabeth": "F", "Nora": "F",
    "Julie": "F", "Alice": "F", "Joanna": "F", "Lydia": "F", "Caitlin": "F",
    "Bonnie": "F", "Erin": "F", "Chloe": "F", "Dina": "F", "Kim": "F",
    "Leslie": "F", "ginger": "F", "Ginger": "F", "Aurora": "F", "Kristen": "F",

    # Secundários recorrentes (Masculinos)
    "Gunther": "M", "Richard": "M", "Mike": "M", "Jack": "M", "Ben": "M",
    "Pete": "M", "David": "M", "Paul": "M", "Frank": "M", "Tag": "M",
    "Joshua": "M", "Gary": "M", "Paolo": "M", "Eddie": "M", "Mark": "M",
    "Barry": "M", "Treeger": "M", "Gavin": "M", "Danny": "M", "Will": "M",
    "Leonard": "M", "parker": "M", "Parker": "M", "Steve": "M", "stu": "M",
    "Stu": "M", "Doug": "M", "Terry": "M", "Allesandro": "M", "Wayne": "M",
}

# ==========================================
# CAMADA 2: Títulos e papéis com gênero explícito no nome
# ==========================================
FEMALE_KEYWORDS = [
    "mrs.", "ms.", "miss", "woman", "girl", "waitress", "hostess",
    "nurse", "mother", "mom", "grandmother", "sister", "lady",
    "stewardess", "actress", "bride", "her", "she"
]
MALE_KEYWORDS = [
    "mr.", "man", "guy", "boy", "waiter", "policeman", "fireman",
    "father", "dad", "grandfather", "brother", "gentleman", "actor",
    "groom", "bellhop", "doorman"
]

def get_lexical_gender(speaker_name: str):
    """Tenta resolver o gênero pelo dicionário de personagens ou palavras-chave."""
    spk_clean = str(speaker_name).strip()
    if spk_clean in KNOWN_SPEAKERS_GENDER:
        return KNOWN_SPEAKERS_GENDER[spk_clean]

    spk_lower = spk_clean.lower()
    for kw in FEMALE_KEYWORDS:
        if kw in spk_lower.split() or spk_lower.startswith(kw):
            return "F"
    for kw in MALE_KEYWORDS:
        if kw in spk_lower.split() or spk_lower.startswith(kw):
            return "M"
    return None  # Vai para a Camada 3 (Acústica)


def classify_and_filter_gender(
    df: pd.DataFrame,
    split_name: str,
    drop_unknown: bool = False,
) -> pd.DataFrame:
    """Add 'gender' column ('F'/'M') using get_lexical_gender and optionally drop unresolved rows."""
    out = df.copy()
    out["gender"] = out["speaker"].map(get_lexical_gender)
    n_total = len(out)
    n_f = int((out["gender"] == "F").sum())
    n_m = int((out["gender"] == "M").sum())
    n_unk = int(out["gender"].isna().sum())

    if drop_unknown:
        out = out.loc[out["gender"].isin(["F", "M"])].reset_index(drop=True)
        print(
            f"[{split_name}] Gender classification: F={n_f:,}, M={n_m:,} | "
            f"Removed unresolved: {n_unk:,}/{n_total:,} -> Retained: {len(out):,} ({100.0 * len(out) / max(1, n_total):.2f}%)"
        )
    else:
        print(
            f"[{split_name}] Gender classification: F={n_f:,}, M={n_m:,}, Unresolved={n_unk:,}/{n_total:,} (kept for eval)"
        )
    return out


# Apply to raw splits (removing unresolved speakers from raw_train_set)
if isinstance(raw_train_set, pd.DataFrame):
    raw_train_set = classify_and_filter_gender(raw_train_set, "raw_train_set", drop_unknown=True)
if isinstance(raw_val_set, pd.DataFrame):
    raw_val_set = classify_and_filter_gender(raw_val_set, "raw_val_set", drop_unknown=False)
if isinstance(raw_test_set, pd.DataFrame):
    raw_test_set = classify_and_filter_gender(raw_test_set, "raw_test_set", drop_unknown=False)

# If clean_train_set was already created in session, filter it as well
if "clean_train_set" in globals() and isinstance(clean_train_set, pd.DataFrame):
    clean_train_set = classify_and_filter_gender(clean_train_set, "clean_train_set", drop_unknown=True)


[raw_train_set] Gender classification: F=4,533, M=4,732 | Removed unresolved: 0/9,265 -> Retained: 9,265 (100.00%)
[raw_val_set] Gender classification: F=540, M=497, Unresolved=71/1,108 (kept for eval)
[raw_test_set] Gender classification: F=1,207, M=1,345, Unresolved=186/2,738 (kept for eval)


## 4. Parallel Standardization to 16 kHz Mono WAV in `DATA_DIR`
All splits (`train`, `val`, `test`) receive the same basic treatment:
- Convert multi-channel audio to mono by channel averaging.
- Resample to `16000 Hz` if needed.
- Save as 16-bit PCM `.wav` under `DATA_DIR / split / dia{dialogue}_utt{utterance}.wav`.
- Update `audio_path` and `duration` in `raw_train_set`, `raw_val_set`, and `raw_test_set`.
- No laugh removal or denoising is applied here.


In [9]:
def _resample_audio(wav: np.ndarray, orig_sr: int, target_sr: int) -> np.ndarray:
    if orig_sr == target_sr:
        return wav
    try:
        import librosa
        return librosa.resample(wav, orig_sr=orig_sr, target_sr=target_sr).astype(np.float32)
    except ImportError:
        pass
    try:
        from scipy.signal import resample_poly
        divisor = gcd(orig_sr, target_sr)
        return resample_poly(wav, target_sr // divisor, orig_sr // divisor).astype(np.float32)
    except ImportError:
        n_out = int(round(len(wav) * target_sr / orig_sr))
        x_old = np.linspace(0.0, 1.0, num=len(wav), endpoint=False)
        x_new = np.linspace(0.0, 1.0, num=n_out, endpoint=False)
        return np.interp(x_new, x_old, wav).astype(np.float32)


def _process_single_audio(task: Tuple[int, str, Path]) -> Tuple[int, Optional[str], float, Optional[str]]:
    idx, src_path, dst_path = task
    if not src_path or not Path(src_path).exists():
        return idx, None, float("nan"), f"Missing source: {src_path}"

    try:
        if dst_path.exists() and not OVERWRITE_AUDIO:
            info = sf.info(str(dst_path))
            if info.samplerate == TARGET_SR and info.channels == TARGET_CHANNELS:
                return idx, str(dst_path.resolve()), float(info.duration), None

        dst_path.parent.mkdir(parents=True, exist_ok=True)
        src_info = sf.info(str(src_path))

        # Fast path: already 16 kHz, mono, PCM_16 WAV (copyfile avoids extra Drive FUSE metadata RPCs)
        if (
            src_info.samplerate == TARGET_SR
            and src_info.channels == TARGET_CHANNELS
            and src_info.subtype == "PCM_16"
            and Path(src_path).suffix.lower() == ".wav"
        ):
            shutil.copyfile(src_path, dst_path)
            return idx, str(dst_path.resolve()), float(src_info.duration), None

        wav, sr = sf.read(str(src_path), dtype="float32", always_2d=True)
        if wav.shape[1] > 1:
            wav = wav.mean(axis=1)
        else:
            wav = wav[:, 0]

        if sr != TARGET_SR:
            wav = _resample_audio(wav, sr, TARGET_SR)

        wav = np.clip(wav, -1.0, 1.0)
        sf.write(str(dst_path), wav, TARGET_SR, subtype="PCM_16")
        duration = float(len(wav) / TARGET_SR)
        return idx, str(dst_path.resolve()), duration, None
    except Exception as exc:
        return idx, None, float("nan"), str(exc)


def export_split_16k_mono(
    df: pd.DataFrame,
    split_name: str,
    data_dir: Path = DATA_DIR,
    max_workers: int = MAX_WORKERS,
    overwrite: bool = OVERWRITE_AUDIO,
) -> pd.DataFrame:
    """Standardize all audio in a split to 16 kHz mono WAV in parallel (fast-skips existing CSV or WAVs on Drive)."""
    csv_path = Path(data_dir) / f"raw_{split_name}_set.csv"
    if not overwrite and df is None and all_clean_outputs_cached(data_dir):
        print(f"[{split_name}] Skipping 16k mono export — cleaned pipeline outputs already cached")
        return None
    if csv_path.exists() and not overwrite:
        cached_df = pd.read_csv(csv_path)
        valid_paths = cached_df["audio_path"].dropna().astype(str).tolist()
        if len(cached_df) > 0 and len(valid_paths) == len(cached_df):
            spot_check = valid_paths[:5] + valid_paths[-5:]
            if all(Path(p).exists() for p in spot_check) or all_clean_outputs_cached(data_dir):
                print(f"[{split_name}] Skipping 16k mono export — loaded cached {csv_path} ({len(cached_df):,} rows)")
                return cached_df

    out_df = df.copy().reset_index(drop=True)
    split_audio_dir = Path(data_dir) / split_name
    split_audio_dir.mkdir(parents=True, exist_ok=True)

    # Single directory listing on Drive instead of 9k+ individual exists()/sf.info() calls
    existing_files = set() if overwrite else {p.name for p in split_audio_dir.glob("*.wav")}

    new_paths = [None] * len(out_df)
    durations = [float("nan")] * len(out_df)
    tasks = []

    for idx, row in out_df.iterrows():
        fname = f"dia{int(row['dialogue'])}_utt{int(row['utterance'])}.wav"
        dst_path = split_audio_dir / fname
        if fname in existing_files:
            new_paths[idx] = str(dst_path.resolve())
            dur = row.get("duration", row.get("time", float("nan")))
            durations[idx] = float(dur) if pd.notna(dur) else float("nan")
        else:
            tasks.append((idx, row["audio_path"], dst_path))

    if existing_files:
        print(f"[{split_name}] Found {len(out_df) - len(tasks):,}/{len(out_df):,} WAVs already in {split_audio_dir}; processing {len(tasks):,} remaining")

    errors = []
    if tasks:
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = [executor.submit(_process_single_audio, t) for t in tasks]
            for fut in tqdm(as_completed(futures), total=len(futures), desc=f"Exporting {split_name} (16k mono)"):
                idx, out_path, dur, err = fut.result()
                new_paths[idx] = out_path
                durations[idx] = dur
                if err is not None:
                    errors.append((idx, err))

    out_df["source_audio_path"] = out_df["audio_path"]
    out_df["audio_path"] = new_paths
    out_df["duration"] = durations

    # Basic hygiene: drop rows whose audio failed or is missing
    valid_mask = out_df["audio_path"].notna() & out_df["emotion"].notna()
    dropped = int((~valid_mask).sum())
    if dropped or errors:
        print(f"[{split_name}] Dropped {dropped} invalid rows ({len(errors)} audio errors)")
        for idx, err in errors[:5]:
            print(f"  - row {idx}: {err}")
    out_df = out_df.loc[valid_mask].reset_index(drop=True)

    out_df.to_csv(csv_path, index=False)
    print(f"[{split_name}] Saved {len(out_df):,} rows -> {csv_path}")
    return out_df


In [10]:
raw_train_set = export_split_16k_mono(raw_train_set, "train", DATA_DIR)
raw_val_set = export_split_16k_mono(raw_val_set, "val", DATA_DIR)
raw_test_set = export_split_16k_mono(raw_test_set, "test", DATA_DIR)


[train] Skipping 16k mono export — loaded cached /content/agora_work/meld_16k_mono/raw_train_set.csv (9,265 rows)
[val] Skipping 16k mono export — loaded cached /content/agora_work/meld_16k_mono/raw_val_set.csv (1,108 rows)
[test] Skipping 16k mono export — loaded cached /content/agora_work/meld_16k_mono/raw_test_set.csv (2,738 rows)


## 5. Verification & Split Summary


In [11]:
def verify_split(df: Optional[pd.DataFrame], name: str, sample_check: int = 100) -> None:
    if df is None or len(df) == 0:
        print(f"=== {name.upper()} — skipped (cleaned pipeline outputs already cached) ===")
        return
    assert df["audio_path"].isna().sum() == 0, f"{name}: NaN audio_path found"
    assert df["emotion"].isna().sum() == 0, f"{name}: NaN emotion found"

    check_paths = df["audio_path"].head(sample_check).tolist()
    if not OVERWRITE_AUDIO and all_clean_outputs_cached(DATA_DIR) and not all(Path(p).exists() for p in check_paths[:5]):
        print(f"=== {name.upper()} ({len(df):,} samples) — skipped raw WAV spot-check (cleaned outputs cached) ===")
        return

    bad_sr, bad_ch = 0, 0
    for p in check_paths:
        info = sf.info(p)
        if info.samplerate != TARGET_SR:
            bad_sr += 1
        if info.channels != TARGET_CHANNELS:
            bad_ch += 1

    print(f"=== {name.upper()} ({len(df):,} samples) ===")
    print(f"Checked {len(check_paths)} files -> bad_sr={bad_sr}, bad_ch={bad_ch}")
    print(f"Duration (s): min={df['duration'].min():.2f}, median={df['duration'].median():.2f}, max={df['duration'].max():.2f}")
    print("Emotions:")
    print(df["emotion"].value_counts().to_string())
    print()


verify_split(raw_train_set, "raw_train_set")
verify_split(raw_val_set, "raw_val_set")
verify_split(raw_test_set, "raw_test_set")

if all(isinstance(d, pd.DataFrame) and len(d) > 0 for d in (raw_train_set, raw_val_set, raw_test_set)):
    # Check dialogue-level overlap across splits (official MELD splits should be disjoint)
    train_dias = set(raw_train_set["dialogue"].unique())
    val_dias = set(raw_val_set["dialogue"].unique())
    test_dias = set(raw_test_set["dialogue"].unique())
    print("Dialogue counts -> train:", len(train_dias), "val:", len(val_dias), "test:", len(test_dias))

=== RAW_TRAIN_SET (9,265 samples) ===
Checked 100 files -> bad_sr=0, bad_ch=0
Duration (s): min=0.06, median=2.47, max=41.05
Emotions:
emotion
neutral     4304
joy         1635
surprise    1131
anger       1044
sadness      639
fear         258
disgust      254

=== RAW_VAL_SET (1,108 samples) ===
Checked 100 files -> bad_sr=0, bad_ch=0
Duration (s): min=0.06, median=2.44, max=28.54
Emotions:
emotion
neutral     469
joy         163
anger       153
surprise    150
sadness     111
fear         40
disgust      22

=== RAW_TEST_SET (2,738 samples) ===
Checked 100 files -> bad_sr=0, bad_ch=0
Duration (s): min=0.13, median=2.52, max=304.96
Emotions:
emotion
neutral     1315
joy          425
anger        361
surprise     297
sadness      217
disgust       70
fear          53

Dialogue counts -> train: 1032 val: 114 test: 280


## 6. EMO-TAG Single-Utterance Cleaning & Background Laughter Removal
Implements the exact 5-step single-utterance audio cleaning pipeline from **EMO-TAG** (*Finding Emotions Where They Belong: Rethinking Audio Emotion Recognition through Masked Temporal Affective Grounding*, Mohamed, Hansen, & Tan, 2026 — `Section 3.1`, `Appendix A.2`, and `Figure 4`):

1. **Input & Transcript Normalization (`Appendix A.2`):**
   - Process audio as `16 kHz` mono waveforms.
   - Lowercase reference transcripts, strip punctuation except apostrophes (`'`), and normalize whitespace.
2. **VAD Segmentation (`Section 3.1`, `Figure 4` Step 2):**
   - Segment each utterance into candidate **speech** and **non-speech** regions using **rVAD** (Tan et al., 2020) and reject empty/inaudible recordings.
3. **First-Pass WhisperX Alignment & Unrelated Speech Removal (`Section 3.1`, `Appendix A.2`, `Figure 4` Step 3):**
   - Transcribe each candidate speech region with **WhisperX** (`whisper-large-v3`) and run forced alignment with the ground-truth transcript.
   - Discard any speech segment that has **zero token overlap** with the ground-truth utterance (removing other speakers / overlapping dialogue).
4. **High-Energy Laughter Removal in Non-Speech Regions (`Section 3.1`, `Appendix A.2`, `Figure 4` Step 4):**
   - For non-speech intervals between selected speech regions, compute **RMS level (dBFS)**, **peak amplitude (dBFS)**, frame-level energy variation, and proportion of voiced frames.
   - **Retain** intervals as natural pauses or sighs **only when `RMS <= -35 dBFS` and `Peak <= -25 dBFS`**.
   - **Remove** high-energy non-speech regions (`RMS > -35 dBFS` or `Peak > -25 dBFS`, i.e., sitcom background laughter tracks).
   - If selected speech regions and retained pause/sigh intervals are contiguous, crop a single continuous span; otherwise concatenate the retained regions.
5. **Second-Pass WhisperX Forced Alignment, Boundary Trimming & Edit-Distance Gate (`Section 3.1`, `Appendix A.2`, `Figure 4` Step 5):**
   - Run **WhisperX** forced alignment again on the retained waveform and crop from the start of the first aligned word to the end of the last aligned word (with small side margins), accepting the refined crop only if it preserves the same transcription.
   - **Reject the sample** if the final aligned transcript has **$\ge 2$ consecutive insertions** (extra words not in the reference utterance) or **$\ge 3$ replacements** (substituted/misrecognized words), or if the resulting audio is empty/inaudible.


In [12]:
# Optional dependency setup for WhisperX and rVAD (run if not already installed)
import importlib.util
import sys

RVAD_REPO_DIR = Path("/home/fuganti/Projects/OPERA/third_party/rVAD")

def ensure_emo_tag_dependencies(install_whisperx: bool = False, clone_rvad: bool = True) -> None:
    if clone_rvad and not RVAD_REPO_DIR.exists():
        try:
            RVAD_REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
            subprocess.run(
                ["git", "clone", "--depth", "1", "https://github.com/zhenghuatan/rVAD.git", str(RVAD_REPO_DIR)],
                check=True,
            )
            print(f"Cloned rVAD into {RVAD_REPO_DIR}")
        except Exception as exc:
            print(f"Note: rVAD git clone skipped ({exc}); using built-in rVAD-compatible segmenter.")

    if RVAD_REPO_DIR.exists() and str(RVAD_REPO_DIR) not in sys.path:
        sys.path.insert(0, str(RVAD_REPO_DIR))

    if install_whisperx and importlib.util.find_spec("whisperx") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "whisperx"], check=True)


In [13]:
from dataclasses import dataclass, asdict
import difflib
import re
from typing import Any, List, Sequence

# EMO-TAG Appendix A.2 & Section 3.1 exact constants
EMO_TAG_SR = 16000
PAUSE_MAX_RMS_DBFS = -35.0          # Appendix A.2: retain pauses/sighs when RMS <= -35 dBFS
PAUSE_MAX_PEAK_DBFS = -25.0         # Appendix A.2: retain pauses/sighs when Peak <= -25 dBFS
BOUNDARY_MARGIN_SEC = 0.05          # Appendix A.2: small margins on both sides of first/last aligned word
MAX_CONSECUTIVE_INSERTIONS = 2      # Section 3.1: reject if >= 2 consecutive insertions
MAX_REPLACEMENTS = 3                # Section 3.1: reject if >= 3 replacements (substitutions)
MIN_SPEECH_DURATION_SEC = 0.15      # Minimum audible speech span duration

CLEAN_DATA_DIR = DATA_DIR / "emo_tag_clean"
CLEAN_DATA_DIR.mkdir(parents=True, exist_ok=True)


@dataclass
class IntervalMetrics:
    start_sec: float
    end_sec: float
    is_speech: bool
    rms_dbfs: float
    peak_dbfs: float
    frame_energy_var: float
    voiced_frame_ratio: float


def normalize_emo_tag_transcript(text: str) -> str:
    """Appendix A.2: lowercase, remove punctuation except apostrophes, and normalize whitespace."""
    if not isinstance(text, str):
        return ""
    # Normalize curly apostrophes to standard ASCII apostrophe
    text = text.replace("’", "'").replace("‘", "'").lower()
    # Remove all punctuation/symbols except word characters, whitespace, and apostrophes
    text = re.sub(r"[^a-z0-9'\s]+", " ", text)
    # Collapse repeated apostrophes or standalone apostrophes
    text = re.sub(r"(?<!\w)'|'(?!\w)", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def tokenize_emo_tag(text: str) -> List[str]:
    norm = normalize_emo_tag_transcript(text)
    return norm.split() if norm else []


def compute_interval_acoustic_metrics(
    wav: np.ndarray,
    sr: int = EMO_TAG_SR,
    start_sec: float = 0.0,
    end_sec: Optional[float] = None,
    is_speech: bool = False,
) -> IntervalMetrics:
    """Appendix A.2: compute RMS (dBFS), peak (dBFS), frame-level energy variation, and voiced frame ratio."""
    eps = 1e-10
    if end_sec is None:
        end_sec = len(wav) / float(sr)
    if len(wav) == 0:
        return IntervalMetrics(start_sec, end_sec, is_speech, -120.0, -120.0, 0.0, 0.0)

    rms = float(np.sqrt(np.mean(np.square(wav, dtype=np.float64))))
    peak = float(np.max(np.abs(wav)))
    rms_dbfs = float(20.0 * np.log10(max(rms, eps)))
    peak_dbfs = float(20.0 * np.log10(max(peak, eps)))

    # Frame-level analysis (25 ms window, 10 ms hop at 16 kHz)
    win_len = int(0.025 * sr)
    hop_len = int(0.010 * sr)
    if len(wav) < win_len:
        frames = wav[None, :]
    else:
        n_frames = 1 + (len(wav) - win_len) // hop_len
        frames = np.lib.stride_tricks.sliding_window_view(wav, win_len)[::hop_len][:n_frames]

    frame_rms_db = 20.0 * np.log10(np.sqrt(np.mean(np.square(frames, dtype=np.float64), axis=1)) + eps)
    frame_energy_var = float(np.var(frame_rms_db)) if len(frame_rms_db) > 1 else 0.0

    # Zero-crossing rate + autocorrelation voicing estimate per frame
    zcr = np.mean(np.abs(np.diff(np.signbit(frames), axis=1)), axis=1)
    voiced_mask = (frame_rms_db > -45.0) & (zcr < 0.25)
    voiced_frame_ratio = float(np.mean(voiced_mask)) if len(voiced_mask) > 0 else 0.0

    return IntervalMetrics(
        start_sec=start_sec,
        end_sec=end_sec,
        is_speech=is_speech,
        rms_dbfs=rms_dbfs,
        peak_dbfs=peak_dbfs,
        frame_energy_var=frame_energy_var,
        voiced_frame_ratio=voiced_frame_ratio,
    )


def is_natural_pause_or_sigh(
    metrics: IntervalMetrics,
    max_rms_dbfs: float = PAUSE_MAX_RMS_DBFS,
    max_peak_dbfs: float = PAUSE_MAX_PEAK_DBFS,
) -> bool:
    """Appendix A.2: retain intervening non-speech intervals as natural pauses/sighs iff RMS <= -35 dBFS and Peak <= -25 dBFS."""
    return (metrics.rms_dbfs <= max_rms_dbfs) and (metrics.peak_dbfs <= max_peak_dbfs)


def rvad_frame_decisions(wav: np.ndarray, sr: int = EMO_TAG_SR) -> np.ndarray:
    """
    Run rVAD (Tan et al., 2020) frame-level voice activity detection (10 ms frame shift).
    Uses official speechproc/rVADfast if present, otherwise uses an rVAD-equivalent
    two-pass spectral flatness + high-pass energy + hangover detector.
    """
    if len(wav) < int(0.025 * sr):
        return np.zeros(0, dtype=np.int32)

    # Try official rVAD implementation if cloned in RVAD_REPO_DIR
    try:
        import speechproc
        winlen, ovrlen = 0.025, 0.010
        pre_coef, nfilter, nftt = 0.97, 20, 512
        ftThres, vadThres = 0.5, 0.4
        nfr10 = int(np.floor((len(wav) - (winlen - ovrlen) * sr) / (ovrlen * sr)))
        if nfr10 > 0:
            ft, flen, fsh10, nfr10 = speechproc.sflux(wav, sr, winlen, ovrlen, nftt)
            pv01 = np.zeros(nfr10)
            pv01[np.less_equal(ft, ftThres)] = 1
            pitch = copy_pv = pv01.copy()
            pvblk = speechproc.pitchblockdetect(pv01, pitch, nfr10, 1)
            fdata = speechproc.highpass(wav, sr)
            noise_samp, noise_seg, n_noise_samp = speechproc.snre_highenergy(
                fdata, nfr10, flen, fsh10, np.round( flen * 0.5 ), pv01, pvblk
            )
            for j in range(n_noise_samp):
                fdata[range(int(noise_samp[j, 0]), int(noise_samp[j, 1]) + 1)] = 0
            vad_seg = speechproc.snre_vad(
                fdata, nfr10, flen, fsh10, np.round(flen * 0.5), pv01, pvblk, vadThres
            )
            return np.asarray(vad_seg, dtype=np.int32)
    except Exception:
        pass

    # Built-in rVAD-equivalent two-pass segment-based spectral flatness + SNR-energy detector
    win_len = int(0.025 * sr)
    hop_len = int(0.010 * sr)
    n_frames = 1 + (len(wav) - win_len) // hop_len
    if n_frames <= 0:
        return np.zeros(0, dtype=np.int32)

    # Pre-emphasis / high-pass filtering (removes DC & low-frequency rumble < 60 Hz)
    hp_wav = np.append(wav[0], wav[1:] - 0.97 * wav[:-1])
    frames = np.lib.stride_tricks.sliding_window_view(hp_wav, win_len)[::hop_len][:n_frames]
    window = np.hamming(win_len).astype(np.float32)
    spec = np.abs(np.fft.rfft(frames * window[None, :], n=512, axis=1)) + 1e-10

    # Spectral flatness (geometric mean / arithmetic mean)
    geo_mean = np.exp(np.mean(np.log(spec), axis=1))
    arith_mean = np.mean(spec, axis=1)
    flatness = geo_mean / arith_mean

    # Short-time frame energy (dBFS)
    raw_frames = np.lib.stride_tricks.sliding_window_view(wav, win_len)[::hop_len][:n_frames]
    rms_db = 20.0 * np.log10(np.sqrt(np.mean(np.square(raw_frames, dtype=np.float64), axis=1)) + 1e-10)

    # Pass 1: estimate noise floor from high-flatness / low-energy frames
    noise_floor_db = float(np.percentile(rms_db, 15))
    snr_db = rms_db - max(noise_floor_db, -65.0)
    speech_frames = (rms_db > -45.0) & ((snr_db > 6.0) | ((flatness < 0.45) & (rms_db > -38.0)))

    # Morphological hangover smoothing (bridge < 150 ms gaps, drop < 80 ms spikes)
    decisions = speech_frames.astype(np.int32)
    # Bridge short gaps (<= 15 frames = 150 ms)
    idx_ones = np.where(decisions == 1)[0]
    if len(idx_ones) > 1:
        for a, b in zip(idx_ones[:-1], idx_ones[1:]):
            if 1 < (b - a) <= 15:
                decisions[a:b] = 1
    # Remove isolated micro-bursts (< 8 frames = 80 ms)
    i = 0
    while i < len(decisions):
        if decisions[i] == 1:
            j = i
            while j < len(decisions) and decisions[j] == 1:
                j += 1
            if (j - i) < 8:
                decisions[i:j] = 0
            i = j
        else:
            i += 1

    return decisions


def segment_audio_with_rvad(
    wav: np.ndarray,
    sr: int = EMO_TAG_SR,
) -> List[IntervalMetrics]:
    """Section 3.1 Step 2: Partition audio into speech and non-speech regions using rVAD."""
    total_dur = len(wav) / float(sr)
    decisions = rvad_frame_decisions(wav, sr=sr)
    if len(decisions) == 0 or not np.any(decisions == 1):
        return []

    hop_sec = 0.010
    intervals: List[IntervalMetrics] = []
    n = len(decisions)
    i = 0
    while i < n:
        state = bool(decisions[i] == 1)
        j = i
        while j < n and (decisions[j] == 1) == state:
            j += 1
        start_sec = round(i * hop_sec, 4)
        end_sec = round(min(total_dur, j * hop_sec if j < n else total_dur), 4)
        s_idx = int(round(start_sec * sr))
        e_idx = int(round(end_sec * sr))
        seg_wav = wav[s_idx:e_idx]
        if len(seg_wav) > 0:
            intervals.append(
                compute_interval_acoustic_metrics(
                    seg_wav, sr=sr, start_sec=start_sec, end_sec=end_sec, is_speech=state
                )
            )
        i = j

    return intervals


def compute_word_edit_diagnostics(
    hyp_tokens: Sequence[str],
    ref_tokens: Sequence[str],
) -> Dict[str, Any]:
    """
    Section 3.1 Step 5: Compare WhisperX transcript tokens against ground-truth reference tokens.
    Computes:
      - token_overlap_count: number of words matching reference
      - max_consecutive_insertions: max consecutive extra words not in reference (reject if >= 2)
      - total_replacements: total substituted/replaced words (reject if >= 3)
    """
    ref_set = set(ref_tokens)
    overlap_count = sum(1 for tok in hyp_tokens if tok in ref_set)

    matcher = difflib.SequenceMatcher(a=list(ref_tokens), b=list(hyp_tokens), autojunk=False)
    max_consec_ins = 0
    total_replacements = 0

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == "insert":
            ins_len = j2 - j1
            if ins_len > max_consec_ins:
                max_consec_ins = ins_len
        elif tag == "replace":
            ref_len = i2 - i1
            hyp_len = j2 - j1
            replaced = min(ref_len, hyp_len)
            total_replacements += replaced
            extra_ins = max(0, hyp_len - ref_len)
            if extra_ins > max_consec_ins:
                max_consec_ins = extra_ins

    reject_insertions = max_consec_ins >= MAX_CONSECUTIVE_INSERTIONS
    reject_replacements = total_replacements >= MAX_REPLACEMENTS

    return {
        "token_overlap_count": int(overlap_count),
        "max_consecutive_insertions": int(max_consec_ins),
        "total_replacements": int(total_replacements),
        "reject_insertions": bool(reject_insertions),
        "reject_replacements": bool(reject_replacements),
        "passes_edit_gate": bool(not reject_insertions and not reject_replacements and overlap_count > 0),
    }


In [14]:
class EmoTagWhisperXCleaner:
    """
    Full EMO-TAG Single-Utterance Audio Cleaner (Mohamed et al., 2026, Section 3.1 & Appendix A.2).
    Combines rVAD segmentation, high-energy non-speech laughter removal (RMS <= -35 dBFS, Peak <= -25 dBFS),
    two-pass WhisperX (whisper-large-v3) transcription + forced alignment, boundary trimming,
    and word-level edit-distance rejection (>= 2 consecutive insertions or >= 3 replacements).
    """

    def __init__(
        self,
        whisper_model_name: str = "large-v3",
        device: Optional[str] = None,
        compute_type: Optional[str] = None,
        language: str = "en",
    ) -> None:
        self.whisper_model_name = whisper_model_name
        self.language = language
        self.device = device
        self.compute_type = compute_type
        self.asr_model = None
        self.align_model = None
        self.align_metadata = None

    def _lazy_init_whisperx(self) -> None:
        if self.asr_model is not None and self.align_model is not None:
            return
        import torch
        import whisperx

        if self.device is None:
            self.device = "cuda" if torch.cuda.is_available() else "cpu"
        if self.compute_type is None:
            self.compute_type = "float16" if self.device == "cuda" else "int8"

        print(f"Loading WhisperX ({self.whisper_model_name}) on {self.device} ({self.compute_type})...")
        self.asr_model = whisperx.load_model(
            self.whisper_model_name,
            device=self.device,
            compute_type=self.compute_type,
            language=self.language,
        )
        self.align_model, self.align_metadata = whisperx.load_align_model(
            language_code=self.language,
            device=self.device,
        )

    def transcribe_and_align(
        self,
        wav: np.ndarray,
        sr: int = EMO_TAG_SR,
        reference_text: Optional[str] = None,
    ) -> Dict[str, Any]:
        """Transcribe a 16 kHz float32 waveform with WhisperX and run forced alignment."""
        self._lazy_init_whisperx()
        import whisperx

        audio_f32 = wav.astype(np.float32)
        asr_res = self.asr_model.transcribe(audio_f32, batch_size=1, language=self.language)
        segments = asr_res.get("segments", [])
        asr_text = " ".join(seg.get("text", "").strip() for seg in segments).strip()

        # Align recognized segments (or align reference transcript if provided for boundary trimming)
        align_input = segments
        if not align_input and reference_text:
            align_input = [{"text": reference_text, "start": 0.0, "end": float(len(audio_f32) / sr)}]

        word_segments: List[Dict[str, Any]] = []
        if align_input:
            try:
                aligned = whisperx.align(
                    align_input,
                    self.align_model,
                    self.align_metadata,
                    audio_f32,
                    self.device,
                    return_char_alignments=False,
                )
                word_segments = [
                    w for w in aligned.get("word_segments", [])
                    if isinstance(w, dict) and w.get("start") is not None and w.get("end") is not None
                ]
            except Exception:
                word_segments = []

        return {
            "asr_text": asr_text,
            "asr_tokens": tokenize_emo_tag(asr_text),
            "word_segments": word_segments,
        }

    def clean_utterance(
        self,
        src_audio_path: str,
        reference_transcript: str,
        dst_audio_path: Optional[Path] = None,
    ) -> Dict[str, Any]:
        """
        Execute EMO-TAG Steps 1–5 on a single MELD utterance.
        Returns a structured dictionary with status ('retained' or rejection reason),
        cleaned waveform / path, and laughter removal diagnostics.
        """
        ref_tokens = tokenize_emo_tag(reference_transcript)
        if not ref_tokens:
            return {"retained": False, "reject_reason": "empty_reference_transcript"}

        if not src_audio_path or not Path(src_audio_path).exists():
            return {"retained": False, "reject_reason": "missing_audio_file"}

        # Step 1: Load as 16 kHz mono float32 waveform (Appendix A.2)
        wav, sr = sf.read(str(src_audio_path), dtype="float32", always_2d=True)
        wav = wav.mean(axis=1) if wav.shape[1] > 1 else wav[:, 0]
        if sr != EMO_TAG_SR:
            wav = _resample_audio(wav, sr, EMO_TAG_SR)
            sr = EMO_TAG_SR

        orig_duration = float(len(wav) / sr)
        if orig_duration < MIN_SPEECH_DURATION_SEC:
            return {"retained": False, "reject_reason": "empty_or_too_short_raw", "orig_duration": orig_duration}

        # Step 2: rVAD segmentation into speech and non-speech regions (Section 3.1)
        intervals = segment_audio_with_rvad(wav, sr=sr)
        speech_indices = [idx for idx, iv in enumerate(intervals) if iv.is_speech]
        if not speech_indices:
            return {"retained": False, "reject_reason": "empty_rvad_no_speech", "orig_duration": orig_duration}

        # Step 3: Transcribe each candidate speech region with WhisperX and check token overlap with reference
        matched_speech_indices: List[int] = []
        unrelated_speech_removed = 0
        for idx in speech_indices:
            iv = intervals[idx]
            s_sample = int(round(iv.start_sec * sr))
            e_sample = int(round(iv.end_sec * sr))
            seg_wav = wav[s_sample:e_sample]
            if len(seg_wav) < int(MIN_SPEECH_DURATION_SEC * sr):
                continue
            seg_res = self.transcribe_and_align(seg_wav, sr=sr)
            overlap = sum(1 for tok in seg_res["asr_tokens"] if tok in set(ref_tokens))
            if overlap > 0:
                matched_speech_indices.append(idx)
            else:
                unrelated_speech_removed += 1

        if not matched_speech_indices:
            return {
                "retained": False,
                "reject_reason": "unrelated_speech_no_token_overlap",
                "orig_duration": orig_duration,
                "unrelated_speech_removed": unrelated_speech_removed,
            }

        # Step 4: Inspect intervening non-speech intervals between first and last matched speech region.
        # Keep natural pauses/sighs (RMS <= -35 dBFS and Peak <= -25 dBFS); remove high-energy laughter tracks.
        first_sp = matched_speech_indices[0]
        last_sp = matched_speech_indices[-1]
        matched_set = set(matched_speech_indices)

        retained_indices: List[int] = []
        laughter_intervals_removed = 0
        laughter_seconds_removed = 0.0
        retained_pauses_count = 0

        # Leading/trailing non-speech outside [first_sp, last_sp] is dropped automatically
        for idx, iv in enumerate(intervals):
            if idx < first_sp or idx > last_sp:
                if not iv.is_speech and not is_natural_pause_or_sigh(iv):
                    laughter_intervals_removed += 1
                    laughter_seconds_removed += (iv.end_sec - iv.start_sec)
                continue

            if idx in matched_set:
                retained_indices.append(idx)
            elif not iv.is_speech:
                if is_natural_pause_or_sigh(iv):
                    retained_indices.append(idx)
                    retained_pauses_count += 1
                else:
                    laughter_intervals_removed += 1
                    laughter_seconds_removed += (iv.end_sec - iv.start_sec)
            else:
                # Unrelated speech segment inside utterance span -> remove
                pass

        # Appendix A.2: If selected regions and intervening intervals are continuous, crop a single interval;
        # otherwise concatenate the retained regions.
        is_contiguous = retained_indices == list(range(first_sp, last_sp + 1))
        if is_contiguous:
            s_sample = int(round(intervals[first_sp].start_sec * sr))
            e_sample = int(round(intervals[last_sp].end_sec * sr))
            stage4_wav = wav[s_sample:e_sample]
        else:
            pieces = [
                wav[int(round(intervals[i].start_sec * sr)) : int(round(intervals[i].end_sec * sr))]
                for i in retained_indices
            ]
            stage4_wav = np.concatenate(pieces, axis=0) if pieces else np.zeros(0, dtype=np.float32)

        if len(stage4_wav) < int(MIN_SPEECH_DURATION_SEC * sr):
            return {"retained": False, "reject_reason": "empty_after_laughter_removal", "orig_duration": orig_duration}

        # Step 5: Second-pass WhisperX forced alignment & boundary trimming (Appendix A.2 & Section 3.1)
        pass2_before = self.transcribe_and_align(stage4_wav, sr=sr, reference_text=reference_transcript)
        final_wav = stage4_wav
        word_segs = pass2_before["word_segments"]

        if word_segs:
            crop_start_sec = max(0.0, float(word_segs[0]["start"]) - BOUNDARY_MARGIN_SEC)
            crop_end_sec = min(float(len(stage4_wav) / sr), float(word_segs[-1]["end"]) + BOUNDARY_MARGIN_SEC)
            if crop_end_sec - crop_start_sec >= MIN_SPEECH_DURATION_SEC:
                candidate_crop = stage4_wav[int(round(crop_start_sec * sr)) : int(round(crop_end_sec * sr))]
                pass2_after = self.transcribe_and_align(candidate_crop, sr=sr)
                # Appendix A.2: accept the refined crop only if it preserves the same transcription
                if pass2_after["asr_tokens"] == pass2_before["asr_tokens"] and len(pass2_after["asr_tokens"]) > 0:
                    final_wav = candidate_crop
                    pass2_final = pass2_after
                else:
                    pass2_final = pass2_before
            else:
                pass2_final = pass2_before
        else:
            pass2_final = pass2_before

        # Final audibility & edit-distance gate (>= 2 consecutive insertions or >= 3 replacements -> reject)
        final_metrics = compute_interval_acoustic_metrics(final_wav, sr=sr, is_speech=True)
        if final_metrics.rms_dbfs <= -50.0 or len(final_wav) < int(MIN_SPEECH_DURATION_SEC * sr):
            return {"retained": False, "reject_reason": "inaudible_after_trim", "orig_duration": orig_duration}

        edit_diag = compute_word_edit_diagnostics(pass2_final["asr_tokens"], ref_tokens)
        if not edit_diag["passes_edit_gate"]:
            reason = (
                "consec_insertions>=2"
                if edit_diag["reject_insertions"]
                else ("replacements>=3" if edit_diag["reject_replacements"] else "zero_final_overlap")
            )
            return {
                "retained": False,
                "reject_reason": reason,
                "orig_duration": orig_duration,
                "clean_duration": float(len(final_wav) / sr),
                "asr_transcript": " ".join(pass2_final["asr_tokens"]),
                "ref_transcript": " ".join(ref_tokens),
                **edit_diag,
            }

        clean_duration = float(len(final_wav) / sr)
        out_path_str = None
        if dst_audio_path is not None:
            dst_audio_path = Path(dst_audio_path)
            dst_audio_path.parent.mkdir(parents=True, exist_ok=True)
            sf.write(str(dst_audio_path), np.clip(final_wav, -1.0, 1.0), sr, subtype="PCM_16")
            out_path_str = str(dst_audio_path.resolve())

        return {
            "retained": True,
            "reject_reason": "retained",
            "clean_audio_path": out_path_str,
            "orig_duration": orig_duration,
            "clean_duration": clean_duration,
            "laughter_intervals_removed": int(laughter_intervals_removed),
            "laughter_seconds_removed": round(float(laughter_seconds_removed), 4),
            "unrelated_speech_removed": int(unrelated_speech_removed),
            "retained_pauses_count": int(retained_pauses_count),
            "was_concatenated": bool(not is_contiguous),
            "asr_transcript": " ".join(pass2_final["asr_tokens"]),
            "ref_transcript": " ".join(ref_tokens),
            **edit_diag,
        }


In [15]:
def run_emo_tag_cleaning_on_split(
    df: pd.DataFrame,
    split_name: str,
    cleaner: EmoTagWhisperXCleaner,
    clean_root: Path = CLEAN_DATA_DIR,
    overwrite: bool = False,
    checkpoint_every: int = 100,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Apply the EMO-TAG 5-step cleaning pipeline across a MELD split (`train`, `val`, or `test`).
    - Skips the entire split if `clean_{split}_set.csv` and `emo_tag_audit_{split}.csv` already exist.
    - Resumes from `emo_tag_audit_{split}.partial.csv` if a previous run was interrupted mid-split.
    """
    split_out_dir = Path(clean_root) / split_name
    split_out_dir.mkdir(parents=True, exist_ok=True)
    clean_csv_path = Path(clean_root) / f"clean_{split_name}_set.csv"
    audit_csv_path = Path(clean_root) / f"emo_tag_audit_{split_name}.csv"
    partial_audit_path = Path(clean_root) / f"emo_tag_audit_{split_name}.partial.csv"

    if clean_csv_path.exists() and audit_csv_path.exists() and not overwrite:
        print(f"[{split_name}] Skipping EMO-TAG cleaning — loaded cached {clean_csv_path}")
        return pd.read_csv(clean_csv_path), pd.read_csv(audit_csv_path)

    cues_csv_path = Path(clean_root) / f"clean_{split_name}_set_with_cues.csv"
    if not overwrite and clean_csv_path.exists():
        print(f"[{split_name}] Skipping EMO-TAG cleaning — loaded cached {clean_csv_path}")
        return pd.read_csv(clean_csv_path), pd.DataFrame()
    if not overwrite and cues_csv_path.exists():
        print(f"[{split_name}] Skipping EMO-TAG cleaning — loaded cached {cues_csv_path}")
        return pd.read_csv(cues_csv_path), pd.DataFrame()
    if df is None:
        print(f"[{split_name}] Skipping EMO-TAG cleaning — raw split not loaded (cleaned outputs cached)")
        return pd.DataFrame(), pd.DataFrame()

    # Load partial checkpoint if resuming an interrupted run
    done_lookup: Dict[Tuple[int, int], Dict[str, Any]] = {}
    if partial_audit_path.exists() and not overwrite:
        try:
            partial_df = pd.read_csv(partial_audit_path)
            for _, prow in partial_df.iterrows():
                key = (int(prow["dialogue"]), int(prow["utterance"]))
                # Keep cached result if rejected OR if retained and the clean WAV exists on disk
                if not bool(prow.get("retained", False)) or (
                    pd.notna(prow.get("clean_audio_path")) and Path(str(prow["clean_audio_path"])).exists()
                ):
                    done_lookup[key] = prow.to_dict()
            if done_lookup:
                print(f"[{split_name}] Resuming EMO-TAG cleaning from checkpoint ({len(done_lookup):,} already processed)")
        except Exception:
            done_lookup = {}

    audit_records: List[Dict[str, Any]] = []
    retained_rows: List[Dict[str, Any]] = []

    for step_idx, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc=f"EMO-TAG cleaning [{split_name}]"), 1):
        dia_id = int(row["dialogue"])
        utt_id = int(row["utterance"])
        key = (dia_id, utt_id)
        dst_wav = split_out_dir / f"dia{dia_id}_utt{utt_id}.wav"

        if key in done_lookup:
            audit_row = done_lookup[key]
            res = audit_row
        else:
            res = cleaner.clean_utterance(
                src_audio_path=str(row["audio_path"]),
                reference_transcript=str(row.get("text", "")),
                dst_audio_path=dst_wav,
            )
            audit_row = {
                "dialogue": dia_id,
                "utterance": utt_id,
                "speaker": row.get("speaker"),
                "emotion": row.get("emotion"),
                "raw_audio_path": row.get("audio_path"),
                **res,
            }

        audit_records.append(audit_row)

        if bool(res.get("retained", False)):
            clean_row = row.to_dict()
            clean_row["raw_audio_path"] = row.get("audio_path")
            clean_row["audio_path"] = res["clean_audio_path"]
            clean_row["orig_duration"] = res["orig_duration"]
            clean_row["duration"] = res["clean_duration"]
            clean_row["laughter_intervals_removed"] = res["laughter_intervals_removed"]
            clean_row["laughter_seconds_removed"] = res["laughter_seconds_removed"]
            clean_row["was_concatenated"] = res["was_concatenated"]
            retained_rows.append(clean_row)

        if checkpoint_every > 0 and (step_idx % checkpoint_every == 0):
            pd.DataFrame(audit_records).to_csv(partial_audit_path, index=False)

    clean_df = pd.DataFrame(retained_rows)
    audit_df = pd.DataFrame(audit_records)

    clean_df.to_csv(clean_csv_path, index=False)
    audit_df.to_csv(audit_csv_path, index=False)
    if partial_audit_path.exists():
        partial_audit_path.unlink()

    n_orig = len(df)
    n_ret = len(clean_df)
    n_rem = n_orig - n_ret
    ret_pct = (100.0 * n_ret / n_orig) if n_orig else 0.0
    laugh_trimmed = int((audit_df.get("laughter_intervals_removed", pd.Series(dtype=int)).fillna(0) > 0).sum())

    print(f"\n=== EMO-TAG Filtering Summary [{split_name.upper()}] (cf. Table 7b) ===")
    print(f"Original : {n_orig:,}")
    print(f"Removed  : {n_rem:,}")
    print(f"Retained : {n_ret:,} ({ret_pct:.2f}%)")
    print(f"Clips with high-energy laughter intervals stripped: {laugh_trimmed:,}")
    print("\nBreakdown by status / rejection reason:")
    print(audit_df["reject_reason"].value_counts().to_string())
    print(f"Saved clean split -> {clean_csv_path}")
    print(f"Saved audit log   -> {audit_csv_path}\n")

    return clean_df, audit_df


In [16]:
# Initialize EMO-TAG cleaner (WhisperX large-v3 + rVAD + -35/-25 dBFS laughter filter)
_needs_emo_tag = OVERWRITE_AUDIO or any(
    not (CLEAN_DATA_DIR / f"clean_{sp}_set.csv").exists()
    and not (CLEAN_DATA_DIR / f"clean_{sp}_set_with_cues.csv").exists()
    and isinstance(raw_df, pd.DataFrame)
    for sp, raw_df in (("train", raw_train_set), ("val", raw_val_set), ("test", raw_test_set))
)
if _needs_emo_tag:
    ensure_emo_tag_dependencies(install_whisperx=False, clone_rvad=True)
emo_tag_cleaner = EmoTagWhisperXCleaner(whisper_model_name="large-v3")

# Run on raw_train_set, raw_val_set, and raw_test_set to produce laugh-free MELD splits for eGeMAPSv02
clean_train_set, train_emo_tag_audit = run_emo_tag_cleaning_on_split(raw_train_set, "train", emo_tag_cleaner)
clean_val_set, val_emo_tag_audit = run_emo_tag_cleaning_on_split(raw_val_set, "val", emo_tag_cleaner)
clean_test_set, test_emo_tag_audit = run_emo_tag_cleaning_on_split(raw_test_set, "test", emo_tag_cleaner)

[train] Skipping EMO-TAG cleaning — loaded cached /content/agora_work/meld_16k_mono/emo_tag_clean/clean_train_set.csv
[val] Skipping EMO-TAG cleaning — loaded cached /content/agora_work/meld_16k_mono/emo_tag_clean/clean_val_set.csv
[test] Skipping EMO-TAG cleaning — loaded cached /content/agora_work/meld_16k_mono/emo_tag_clean/clean_test_set.csv


## 7. eGeMAPSv02 Acoustic Feature Extraction (`openSMILE`) & Training Set Integrity Filtering
Extracts the 5 core `ACOUSTIC_CUES` from the training and validation sets using `opensmile` (`eGeMAPSv02`, `Functionals`), then filters the training set to remove:
- Any utterance where **any** of the 5 `ACOUSTIC_CUES` features is `0.0` (or `NaN`).
- Any utterance with duration **`< 0.5s` or `> 20.0s`**.


In [17]:
ACOUSTIC_CUES = {
    # 1. Altura do Pitch (Mediana em semitons - ref 27.5Hz)
    "pitch_level": "F0semitoneFrom27.5Hz_sma3nz_percentile50.0",

    # 2. Dinâmica de Pitch (Range interpercentil P80 - P20 em semitons)
    "pitch_variation": "F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2",

    # 3. Energia Vocal (Mediana perceptual em Sones)
    "vocal_energy": "loudness_sma3_percentile50.0",

    # 4. Taxa de Fala / Ritmo (Proxy temporal do eGeMAPS)
    # Obs: Se notar zeros em clipes curtos pós-WhisperX, use "loudnessPeaksPerSec"
    "speaking_rate": "VoicedSegmentsPerSec",

    # 5. Qualidade / Tensão Vocal (Balanço espectral 50-1000Hz vs 1-5kHz em dB)
    "timbre": "alphaRatioV_sma3nz_amean",
}

ACOUSTIC_FEATURE_COLS = list(ACOUSTIC_CUES.values())
MIN_TRAIN_DURATION_SEC = 0.5
MAX_TRAIN_DURATION_SEC = 20.0

smile_extractor = None


def _get_smile_extractor():
    global smile_extractor
    if smile_extractor is None:
        import opensmile
        smile_extractor = opensmile.Smile(
            feature_set=opensmile.FeatureSet.eGeMAPSv02,
            feature_level=opensmile.FeatureLevel.Functionals,
        )
    return smile_extractor


def _extract_egemaps_for_row(task: Tuple[int, str]) -> Tuple[int, Dict[str, float], Optional[str]]:
    idx, audio_path = task
    if not audio_path or not Path(audio_path).exists():
        return idx, {col: float("nan") for col in ACOUSTIC_FEATURE_COLS}, f"Missing file: {audio_path}"
    try:
        feats_df = _get_smile_extractor().process_file(str(audio_path))
        row_feats = {
            col: float(feats_df[col].iloc[0]) if col in feats_df.columns else float("nan")
            for col in ACOUSTIC_FEATURE_COLS
        }
        return idx, row_feats, None
    except Exception as exc:
        return idx, {col: float("nan") for col in ACOUSTIC_FEATURE_COLS}, str(exc)


def extract_egemaps_cues(
    df: pd.DataFrame,
    split_name: str,
    max_workers: int = MAX_WORKERS,
) -> pd.DataFrame:
    """Extract the 5 ACOUSTIC_CUES (eGeMAPSv02 Functionals) in parallel for a split DataFrame."""
    out = df.copy().reset_index(drop=True)

    # Ensure duration column is populated in seconds
    if "duration" not in out.columns or out["duration"].isna().any():
        if "time" in out.columns:
            out["duration"] = pd.to_numeric(out["time"], errors="coerce")
        else:
            out["duration"] = [
                float(sf.info(str(p)).duration) if p and Path(str(p)).exists() else float("nan")
                for p in out["audio_path"]
            ]

    tasks = [(idx, str(row["audio_path"])) for idx, row in out.iterrows()]
    extracted_rows: List[Optional[Dict[str, float]]] = [None] * len(out)
    errors = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(_extract_egemaps_for_row, t) for t in tasks]
        for fut in tqdm(as_completed(futures), total=len(futures), desc=f"Extracting eGeMAPSv02 [{split_name}]"):
            idx, feat_dict, err = fut.result()
            extracted_rows[idx] = feat_dict
            if err is not None:
                errors.append((idx, err))

    feats_table = pd.DataFrame(extracted_rows)
    for col in ACOUSTIC_FEATURE_COLS:
        out[col] = feats_table[col].values
    for alias, col in ACOUSTIC_CUES.items():
        out[alias] = out[col]

    if errors:
        print(f"[{split_name}] Warning: {len(errors)} extraction errors encountered.")
    return out


def filter_training_set_by_duration_and_nonzero_cues(
    df: pd.DataFrame,
    split_name: str = "train",
    min_dur: float = MIN_TRAIN_DURATION_SEC,
    max_dur: float = MAX_TRAIN_DURATION_SEC,
    delete_rejected_wavs_in_data_dir: bool = True,
) -> pd.DataFrame:
    """
    Check the training set by duration ([min_dur, max_dur]) and non-zero ACOUSTIC_CUES.
    Deletes any training utterance with duration < 0.5s or > 20.0s, or with any ACOUSTIC_CUES == 0.0 / NaN.
    """
    n_initial = len(df)
    dur_series = pd.to_numeric(df["duration"], errors="coerce")
    bad_short = dur_series < min_dur
    bad_long = dur_series > max_dur
    bad_dur = dur_series.isna() | bad_short | bad_long

    zero_or_nan_per_cue: Dict[str, int] = {}
    bad_cues = pd.Series(False, index=df.index)
    for alias, col in ACOUSTIC_CUES.items():
        s = pd.to_numeric(df[col], errors="coerce")
        is_zero_or_nan = s.isna() | (s == 0.0)
        zero_or_nan_per_cue[f"{alias} ({col})"] = int(is_zero_or_nan.sum())
        bad_cues = bad_cues | is_zero_or_nan

    reject_mask = bad_dur | bad_cues
    rejected_df = df.loc[reject_mask]
    filtered_df = df.loc[~reject_mask].reset_index(drop=True)

    # Optionally remove rejected exported .wav files inside DATA_DIR (never outside DATA_DIR)
    deleted_files = 0
    if delete_rejected_wavs_in_data_dir and "DATA_DIR" in globals():
        data_root = Path(DATA_DIR).resolve()
        for wav_str in rejected_df["audio_path"].dropna().unique():
            wav_p = Path(str(wav_str)).resolve()
            if wav_p.exists() and wav_p.is_relative_to(data_root):
                try:
                    wav_p.unlink()
                    deleted_files += 1
                except OSError:
                    pass

    print(f"\n=== Training Set eGeMAPSv02 & Duration Audit [{split_name}] ===")
    print(f"Initial training samples        : {n_initial:,}")
    print(f"Duration < {min_dur}s removed          : {int(bad_short.sum()):,}")
    print(f"Duration > {max_dur}s removed         : {int(bad_long.sum()):,}")
    print("Zero / NaN count per acoustic cue:")
    for k, cnt in zero_or_nan_per_cue.items():
        print(f"  - {k}: {cnt:,}")
    print(f"Total with >=1 zero/NaN feature : {int(bad_cues.sum()):,}")
    print(f"Total removed (union)           : {int(reject_mask.sum()):,} (deleted {deleted_files:,} WAV files in DATA_DIR)")
    print(f"Final retained training samples : {len(filtered_df):,} ({100.0 * len(filtered_df) / max(1, n_initial):.2f}%)\n")

    return filtered_df


# Select active train and val splits (prefers EMO-TAG clean splits if computed, else raw splits)
active_train_df = (
    clean_train_set
    if ("clean_train_set" in globals() and isinstance(clean_train_set, pd.DataFrame) and len(clean_train_set) > 0)
    else raw_train_set
)
active_val_df = (
    clean_val_set
    if ("clean_val_set" in globals() and isinstance(clean_val_set, pd.DataFrame) and len(clean_val_set) > 0)
    else raw_val_set
)
active_out_dir = (
    CLEAN_DATA_DIR
    if ("clean_train_set" in globals() and isinstance(clean_train_set, pd.DataFrame) and len(clean_train_set) > 0)
    else DATA_DIR
)

train_egemaps_csv = Path(active_out_dir) / "train_egemaps.csv"
val_egemaps_csv = Path(active_out_dir) / "val_egemaps.csv"

if train_egemaps_csv.exists() and val_egemaps_csv.exists() and not OVERWRITE_AUDIO:
    print(f"[eGeMAPSv02] Skipping extraction — loading cached {train_egemaps_csv} and {val_egemaps_csv}")
    train_egemaps_df = pd.read_csv(train_egemaps_csv)
    val_egemaps_df = pd.read_csv(val_egemaps_csv)
else:
    # 1. Extract eGeMAPSv02 features for train and val
    train_egemaps_df = extract_egemaps_cues(active_train_df, "train")
    val_egemaps_df = extract_egemaps_cues(active_val_df, "val")

    # 2. Filter training set by duration [0.5s, 20.0s] and strictly non-zero eGeMAPSv02 features
    train_egemaps_df = filter_training_set_by_duration_and_nonzero_cues(train_egemaps_df, "train")

    train_egemaps_df.to_csv(train_egemaps_csv, index=False)
    val_egemaps_df.to_csv(val_egemaps_csv, index=False)
    print(f"Saved train eGeMAPSv02 dataset -> {train_egemaps_csv} ({len(train_egemaps_df):,} rows)")
    print(f"Saved val eGeMAPSv02 dataset   -> {val_egemaps_csv} ({len(val_egemaps_df):,} rows)")

# Update active split references
if "clean_train_set" in globals() and isinstance(clean_train_set, pd.DataFrame) and len(clean_train_set) > 0:
    clean_train_set = train_egemaps_df
    clean_val_set = val_egemaps_df
else:
    raw_train_set = train_egemaps_df
    raw_val_set = val_egemaps_df


[eGeMAPSv02] Skipping extraction — loading cached /content/agora_work/meld_16k_mono/emo_tag_clean/train_egemaps.csv and /content/agora_work/meld_16k_mono/emo_tag_clean/val_egemaps.csv


## 8. Gender-Specific Quartile Classification (`F` vs. `M`) for Acoustic Cues
Computes `Q25` and `Q75` per acoustic cue separately for Female (`"F"`) and Male (`"M"`) speakers on `clean_train_set`:
- **`< Q25`** $\rightarrow$ `low`
- **`> Q75`** $\rightarrow$ `high`
- **`Q25 <= x <= Q75`** $\rightarrow$ `moderate`
- **`timbre`** (`alphaRatioV_sma3nz_amean`) is mapped to:
  - `low` $\rightarrow$ `"low (muffled/soft)"`
  - `moderate` $\rightarrow$ `"moderate (modal)"`
  - `high` $\rightarrow$ `"high (tense/bright)"`


In [18]:
import json

# Ensure 'timbre' alias is mapped to alphaRatioV_sma3nz_amean
ACOUSTIC_CUES = {
    "pitch_level": "F0semitoneFrom27.5Hz_sma3nz_percentile50.0",
    "pitch_variation": "F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2",
    "vocal_energy": "loudness_sma3_percentile50.0",
    "speaking_rate": "VoicedSegmentsPerSec",
    "timbre": "alphaRatioV_sma3nz_amean",
}

TIMBRE_LABEL_MAP: Dict[str, str] = {
    "low": "low (muffled/soft)",
    "moderate": "moderate (modal)",
    "high": "high (tense/bright)",
}


def compute_gender_acoustic_quartiles(
    train_df: pd.DataFrame,
    cues_map: Dict[str, str] = ACOUSTIC_CUES,
) -> Dict[str, Dict[str, Dict[str, float]]]:
    """
    Compute Q25 and Q75 for each acoustic cue on clean_train_set separately for 'F' and 'M'
    (plus '__GLOBAL__' fallback and an acoustic F0 midpoint for unresolved eval speakers).
    """
    if "gender" not in train_df.columns:
        raise ValueError("Column 'gender' ('F'/'M') not found in training DataFrame.")

    quartiles: Dict[str, Dict[str, Dict[str, float]]] = {"F": {}, "M": {}, "__GLOBAL__": {}}

    for gender_key in ("F", "M"):
        sub = train_df.loc[train_df["gender"] == gender_key]
        if len(sub) == 0:
            raise ValueError(f"No samples found for gender={gender_key!r} in training DataFrame.")
        for cue_alias, col_name in cues_map.items():
            src_col = col_name if col_name in sub.columns else cue_alias
            s = pd.to_numeric(sub[src_col], errors="coerce").dropna()
            s = s[s != 0.0]
            quartiles[gender_key][cue_alias] = {
                "q25": float(s.quantile(0.25)),
                "q50": float(s.quantile(0.50)),
                "q75": float(s.quantile(0.75)),
                "count": int(len(s)),
            }

    for cue_alias, col_name in cues_map.items():
        src_col = col_name if col_name in train_df.columns else cue_alias
        s = pd.to_numeric(train_df[src_col], errors="coerce").dropna()
        s = s[s != 0.0]
        quartiles["__GLOBAL__"][cue_alias] = {
            "q25": float(s.quantile(0.25)),
            "q50": float(s.quantile(0.50)),
            "q75": float(s.quantile(0.75)),
            "count": int(len(s)),
        }

    # Midpoint between Male and Female median F0 (Camada 3 acoustic fallback for eval speakers with gender=None)
    f0_m = quartiles["M"]["pitch_level"]["q50"]
    f0_f = quartiles["F"]["pitch_level"]["q50"]
    quartiles["__META__"] = {"f0_gender_boundary_st": float(0.5 * (f0_m + f0_f))}

    return quartiles


def classify_value_by_quartile(
    val: float,
    q25: float,
    q75: float,
    cue_alias: str,
) -> Optional[str]:
    """
    Rule:
      < Q25  -> 'low'
      > Q75  -> 'high'
      else   -> 'moderate'
    For 'timbre', maps low/moderate/high to:
      'low (muffled/soft)' | 'moderate (modal)' | 'high (tense/bright)'
    """
    if val is None or pd.isna(val) or float(val) == 0.0:
        return None
    v = float(val)
    if v < q25:
        tier = "low"
    elif v > q75:
        tier = "high"
    else:
        tier = "moderate"

    if cue_alias == "timbre":
        return TIMBRE_LABEL_MAP[tier]
    return tier


def apply_gender_quartile_classification(
    df: pd.DataFrame,
    quartiles: Dict[str, Dict[str, Dict[str, float]]],
    split_name: str,
    cues_map: Dict[str, str] = ACOUSTIC_CUES,
) -> pd.DataFrame:
    """Classify each acoustic cue into gender-relative quartile tiers on a split DataFrame."""
    out = df.copy().reset_index(drop=True)
    f0_boundary = quartiles.get("__META__", {}).get("f0_gender_boundary_st", 36.0)
    f0_col = cues_map["pitch_level"] if cues_map["pitch_level"] in out.columns else "pitch_level"

    # Resolve effective gender ('F' or 'M') per row; use F0 boundary only if gender is None (Camada 3)
    resolved_genders: List[str] = []
    for _, row in out.iterrows():
        g = row.get("gender")
        if g in ("F", "M"):
            resolved_genders.append(g)
        else:
            f0_val = row.get(f0_col, float("nan"))
            if pd.notna(f0_val) and float(f0_val) > 0.0:
                resolved_genders.append("F" if float(f0_val) >= f0_boundary else "M")
            else:
                resolved_genders.append("__GLOBAL__")
    out["effective_gender"] = resolved_genders

    for cue_alias, col_name in cues_map.items():
        src_col = col_name if col_name in out.columns else cue_alias
        out[cue_alias] = pd.to_numeric(out[src_col], errors="coerce")

        labels: List[Optional[str]] = []
        for idx, row in out.iterrows():
            g_key = out.at[idx, "effective_gender"]
            q_info = quartiles.get(g_key, quartiles["__GLOBAL__"]).get(cue_alias)
            labels.append(
                classify_value_by_quartile(
                    row[cue_alias],
                    q25=q_info["q25"],
                    q75=q_info["q75"],
                    cue_alias=cue_alias,
                )
            )
        out[f"{cue_alias}_cue"] = labels

    print(f"\n=== Gender-Quartile Acoustic Cue Distribution [{split_name}] (N={len(out):,}) ===")
    for cue_alias in cues_map:
        col_cue = f"{cue_alias}_cue"
        counts = out[col_cue].value_counts(dropna=False)
        pcts = (out[col_cue].value_counts(normalize=True, dropna=False) * 100.0).round(2)
        summary = ", ".join(f"{k}: {counts[k]:,} ({pcts[k]}%)" for k in counts.index)
        print(f"  - {cue_alias:<16}: {summary}")

    return out


quartiles_json_path = Path(active_out_dir) / "gender_acoustic_quartiles.json"
train_cues_csv = Path(active_out_dir) / "clean_train_set_with_cues.csv"
val_cues_csv = Path(active_out_dir) / "clean_val_set_with_cues.csv"

if quartiles_json_path.exists() and train_cues_csv.exists() and val_cues_csv.exists() and not OVERWRITE_AUDIO:
    print(f"[Quartiles] Skipping classification — loading cached {train_cues_csv}, {val_cues_csv}, and {quartiles_json_path}")
    GENDER_ACOUSTIC_QUARTILES = json.loads(quartiles_json_path.read_text(encoding="utf-8"))
    clean_train_set = pd.read_csv(train_cues_csv)
    clean_val_set = pd.read_csv(val_cues_csv)
else:
    # 1. Fit F and M quartiles strictly on clean_train_set (or active train_egemaps_df)
    ref_train_df = (
        clean_train_set
        if ("clean_train_set" in globals() and isinstance(clean_train_set, pd.DataFrame) and len(clean_train_set) > 0)
        else train_egemaps_df
    )
    ref_val_df = (
        clean_val_set
        if ("clean_val_set" in globals() and isinstance(clean_val_set, pd.DataFrame) and len(clean_val_set) > 0)
        else val_egemaps_df
    )

    GENDER_ACOUSTIC_QUARTILES = compute_gender_acoustic_quartiles(ref_train_df, ACOUSTIC_CUES)

    # Print fitted F vs M quartile cutoffs
    quartile_rows = []
    for g_key in ("F", "M"):
        for cue_alias, stats in GENDER_ACOUSTIC_QUARTILES[g_key].items():
            quartile_rows.append({
                "gender": g_key,
                "cue": cue_alias,
                "feature": ACOUSTIC_CUES[cue_alias],
                "Q25 (Low <)": round(stats["q25"], 4),
                "Q50 (Median)": round(stats["q50"], 4),
                "Q75 (High >)": round(stats["q75"], 4),
                "N_train": stats["count"],
            })
    quartiles_summary_df = pd.DataFrame(quartile_rows)
    print("=== Clean Train Set Quartile Cutoffs by Gender (F vs. M) ===")
    print(quartiles_summary_df.to_string(index=False))

    # 2. Apply quartile classification to clean_train_set and clean_val_set
    clean_train_set = apply_gender_quartile_classification(ref_train_df, GENDER_ACOUSTIC_QUARTILES, "clean_train_set")
    clean_val_set = apply_gender_quartile_classification(ref_val_df, GENDER_ACOUSTIC_QUARTILES, "clean_val_set")

    # 3. Save quartile JSON and updated CSV manifests
    quartiles_json_path.write_text(json.dumps(GENDER_ACOUSTIC_QUARTILES, indent=2) + "\n", encoding="utf-8")
    clean_train_set.to_csv(train_cues_csv, index=False)
    clean_val_set.to_csv(val_cues_csv, index=False)

    print(f"\nSaved gender quartiles JSON -> {quartiles_json_path}")
    print(f"Saved classified train CSV  -> {train_cues_csv} ({len(clean_train_set):,} rows)")
    print(f"Saved classified val CSV    -> {val_cues_csv} ({len(clean_val_set):,} rows)")


[Quartiles] Skipping classification — loading cached /content/agora_work/meld_16k_mono/emo_tag_clean/clean_train_set_with_cues.csv, /content/agora_work/meld_16k_mono/emo_tag_clean/clean_val_set_with_cues.csv, and /content/agora_work/meld_16k_mono/emo_tag_clean/gender_acoustic_quartiles.json


## 9. One-Shot Save from Local SSD to Google Drive
Copies generated `.csv` / `.json` manifests to `DRIVE_DATA_DIR` and archives cleaned `16 kHz` mono WAVs into a single `meld_16k_mono_clean.tar` bundle for fast Drive upload and future runtime restoration.


In [19]:
def save_local_pipeline_to_drive(
    local_dir: Path = DATA_DIR,
    drive_dir: Optional[Path] = DRIVE_DATA_DIR,
    overwrite: bool = OVERWRITE_AUDIO,
) -> None:
    """Copy missing CSV/JSON outputs and archive clean WAVs once to Google Drive without rewriting existing data."""
    if drive_dir is None:
        print(f"DRIVE_DATA_DIR is None (local mode). All artifacts remain in {local_dir}")
        return

    drive_dir.mkdir(parents=True, exist_ok=True)
    clean_dir = local_dir / "emo_tag_clean"

    # 1. Copy top-level raw CSVs only if missing on Drive (or overwrite=True)
    for p in local_dir.glob("*.csv"):
        dst = drive_dir / p.name
        if overwrite or not dst.exists():
            shutil.copyfile(p, dst)

    # 2. Copy small EMO-TAG, eGeMAPSv02, and quartile CSV/JSON files only if missing on Drive (or overwrite=True)
    if clean_dir.exists():
        drive_clean_dir = drive_dir / "emo_tag_clean"
        drive_clean_dir.mkdir(parents=True, exist_ok=True)
        copied = 0
        for ext in ("*.csv", "*.json"):
            for p in clean_dir.glob(ext):
                dst = drive_clean_dir / p.name
                if overwrite or not dst.exists():
                    shutil.copyfile(p, dst)
                    copied += 1

        # 3. Pack cleaned 16 kHz mono WAV bundle only if the Drive archive does not already exist
        drive_tar = drive_dir / "meld_16k_mono_clean.tar"
        if drive_tar.exists() and not overwrite:
            print(f"[Drive Sync] Skipping tar archive — already registered at {drive_tar} (copied {copied} new metadata files)")
        else:
            local_tar = local_dir.parent / "meld_16k_mono_clean.tar"
            print(f"Packing cleaned audio -> {local_tar}...")
            subprocess.run(
                ["tar", "-cf", str(local_tar), "-C", str(local_dir.parent), local_dir.name],
                check=True,
            )
            partial_tar = drive_tar.with_suffix(".tar.partial")
            try:
                shutil.copyfile(local_tar, partial_tar)
                partial_tar.replace(drive_tar)
            finally:
                partial_tar.unlink(missing_ok=True)
            print(f"Saved archive to Drive -> {drive_tar}")

    print(f"Sync to Drive complete -> {drive_dir}")


save_local_pipeline_to_drive()

[Drive Sync] Skipping tar archive — already registered at /content/drive/MyDrive/AGORA_SER/data/meld_16k_mono/meld_16k_mono_clean.tar (copied 0 new metadata files)
Sync to Drive complete -> /content/drive/MyDrive/AGORA_SER/data/meld_16k_mono


## 10. Batched WhisperX True Syllable & Word Rate Tracking (CMUdict + `g2p_en`, Cleaned Training Set)
Runs batched **WhisperX (`large-v3` + wav2vec2 forced alignment, 16 audios per batch)** (Bain et al., Interspeech 2023) over the full cleaned training set (`clean_train_set_with_cues.csv`) to measure true speaking rate from **word-level timestamps + inter-word pause duration**:
- **Phonetic syllable counting**: Counts ARPAbet vowel nuclei (phonemes carrying stress markers `0`/`1`/`2`) via the **CMU Pronouncing Dictionary (`cmudict` v0.7b**, Weide 1993 / Lenzo & Rudnicky 2014), with **`g2p_en`** (Park & Kim 2019) neural grapheme-to-phoneme fallback for out-of-vocabulary tokens and numerals — zero ad-hoc orthographic exception lists.
- Computes `whisperx_word_count`, `whisperx_syllable_count`, `whisperx_speech_duration`, `whisperx_pause_duration`, `whisperx_utterance_span` (`speech + pause`), `whisperx_syllable_rate` (syllables/s), and `whisperx_word_rate` (words/s).
- Classifies `whisperx_syllable_rate_cue` and `whisperx_word_rate_cue` using **global (gender-agnostic) 25 / 50 / 25 quartiles** (`< Q25 -> low`, `Q25 <= x <= Q75 -> moderate`, `> Q75 -> high`) fitted on the full cleaned training set.
- Appends the new numeric and quartile columns **only inside `clean_train_set_with_cues.csv`** (and atomically syncs that CSV to Drive), preserving all previously registered columns, `gender_acoustic_quartiles.json`, `clean_val_set_with_cues.csv`, `meld_16k_mono_clean.tar`, and audio files untouched.

In [21]:
import gc
import math

WHISPERX_RATE_BATCH_SIZE = 32
WHISPERX_RATE_SILENCE_PAD_SEC = 0.50
WHISPERX_RATE_CHECKPOINT_EVERY_BATCHES = 25
OVERWRITE_WHISPERX_RATE = False

WHISPERX_RATE_COLUMNS = (
    "whisperx_word_count",
    "whisperx_syllable_count",
    "whisperx_speech_duration",
    "whisperx_pause_duration",
    "whisperx_utterance_span",
    "whisperx_syllable_rate",
    "whisperx_syllable_rate_cue",
    "whisperx_word_rate",
    "whisperx_word_rate_cue",
)


def count_arpabet_syllables(phonemes: Sequence[str]) -> int:
    """Count syllables from an ARPAbet phoneme sequence via vowel stress markers (0, 1, 2)."""
    return sum(1 for p in phonemes if isinstance(p, str) and len(p) > 0 and p[-1].isdigit())


class PhoneticSyllableCounter:
    """
    Standardized English phonetic syllable counter backed by:
      1. CMU Pronouncing Dictionary (cmudict v0.7b, Weide 1993 / Lenzo & Rudnicky 2014)
      2. g2p_en neural Grapheme-to-Phoneme fallback (Park & Kim 2019) for OOV words and numbers
    Counts ARPAbet vowel nuclei (phonemes ending in stress digits 0/1/2) with memoization.
    """

    def __init__(
        self,
        cmu_dict: Optional[Dict[str, List[List[str]]]] = None,
        g2p_fn: Any = None,
    ) -> None:
        self._cmu_dict = cmu_dict
        self._g2p_fn = g2p_fn
        self._cache: Dict[str, int] = {}
        self.cmu_hits: int = 0
        self.g2p_hits: int = 0

    def _ensure_lexicon(self) -> Dict[str, List[List[str]]]:
        if self._cmu_dict is None:
            try:
                import cmudict
                self._cmu_dict = cmudict.dict()
            except ImportError:
                import nltk
                nltk.download("cmudict", quiet=True)
                from nltk.corpus import cmudict as nltk_cmudict
                self._cmu_dict = nltk_cmudict.dict()
        return self._cmu_dict

    def _ensure_g2p(self) -> Any:
        if self._g2p_fn is None:
            import nltk
            # NLTK 3.9+ requires the explicit '_eng' averaged perceptron tagger resource for g2p_en
            for resource in ("averaged_perceptron_tagger_eng", "averaged_perceptron_tagger", "cmudict"):
                try:
                    nltk.download(resource, quiet=True)
                except Exception:
                    pass
            from g2p_en import G2p
            self._g2p_fn = G2p()
        return self._g2p_fn

    def count_word(self, word: str) -> int:
        """Return the phonetic syllable count for a single word token (0 for pure punctuation)."""
        if not isinstance(word, str):
            return 0
        cleaned = word.replace("’", "'").replace("‘", "'").strip().lower()
        cleaned = re.sub(r"^[^a-z0-9]+|[^a-z0-9]+$", "", cleaned)
        if not cleaned:
            return 0

        if cleaned in self._cache:
            return self._cache[cleaned]

        cmu = self._ensure_lexicon()

        # 1. Primary lookup in CMU Pronouncing Dictionary (primary pronunciation variant)
        prons = cmu.get(cleaned)
        if prons:
            syl = max(1, count_arpabet_syllables(prons[0]))
            self._cache[cleaned] = syl
            self.cmu_hits += 1
            return syl

        # 2. Hyphenated compounds (e.g. "no-no", "twenty-one") if not listed directly in CMUdict
        if "-" in cleaned:
            parts = [p for p in cleaned.split("-") if p]
            if len(parts) > 1:
                syl = sum(self.count_word(p) for p in parts)
                self._cache[cleaned] = syl
                return syl

        # 3. Neural G2P fallback (g2p_en) for OOV tokens, contractions, slang, and numerals
        g2p = self._ensure_g2p()
        phones = g2p(cleaned)
        syl = max(1, count_arpabet_syllables(phones))
        self._cache[cleaned] = syl
        self.g2p_hits += 1
        return syl


def compute_speech_rate_metrics_from_words(
    word_segments: Sequence[Dict[str, Any]],
    audio_duration: float,
    syllable_counter: PhoneticSyllableCounter,
    reference_text: str = "",
) -> Dict[str, float]:
    """
    Compute true syllable/word rates from WhisperX word timestamps + inter-word pause duration.

    Total utterance span = active merged word duration + sum of inter-word pauses
                         = (last_word.end - first_word.start).
    Falls back cleanly to reference transcript over audio_duration if alignment produces no words.
    """
    dur = float(audio_duration) if (audio_duration is not None and np.isfinite(audio_duration) and audio_duration > 0.0) else float("nan")
    valid_words: List[Tuple[float, float, int]] = []

    for w in word_segments or []:
        if not isinstance(w, dict):
            continue
        token = str(w.get("word", "")).strip()
        syl = syllable_counter.count_word(token)
        if syl <= 0:
            continue
        start = w.get("start")
        end = w.get("end")
        if start is None or end is None:
            continue
        try:
            s_val, e_val = float(start), float(end)
        except (TypeError, ValueError):
            continue
        if not (math.isfinite(s_val) and math.isfinite(e_val)):
            continue
        if math.isfinite(dur):
            s_val = max(0.0, min(dur, s_val))
            e_val = max(0.0, min(dur, e_val))
        if e_val > s_val:
            valid_words.append((s_val, e_val, syl))

    if valid_words:
        valid_words.sort(key=lambda item: (item[0], item[1]))
        word_count = len(valid_words)
        syllable_count = sum(item[2] for item in valid_words)

        # Merge any overlapping word intervals so active speech and pauses partition the span exactly
        merged: List[List[float]] = [[valid_words[0][0], valid_words[0][1]]]
        for s_val, e_val, _ in valid_words[1:]:
            if s_val <= merged[-1][1]:
                if e_val > merged[-1][1]:
                    merged[-1][1] = e_val
            else:
                merged.append([s_val, e_val])

        speech_duration = float(sum(end - start for start, end in merged))
        pause_duration = float(
            sum(max(0.0, merged[k][0] - merged[k - 1][1]) for k in range(1, len(merged)))
        )
        utterance_span = speech_duration + pause_duration
        if utterance_span <= 1e-4 and math.isfinite(dur) and dur > 0.0:
            utterance_span = dur
            speech_duration = dur
            pause_duration = 0.0
    else:
        # Fallback if forced alignment returned no timestamped words
        ref_tokens = tokenize_emo_tag(reference_text)
        word_count = len(ref_tokens)
        syllable_count = sum(syllable_counter.count_word(tok) for tok in ref_tokens)
        if not (math.isfinite(dur) and dur > 0.0) or word_count == 0:
            return {
                "whisperx_word_count": 0,
                "whisperx_syllable_count": 0,
                "whisperx_speech_duration": float("nan"),
                "whisperx_pause_duration": float("nan"),
                "whisperx_utterance_span": float("nan"),
                "whisperx_syllable_rate": float("nan"),
                "whisperx_word_rate": float("nan"),
            }
        speech_duration = dur
        pause_duration = 0.0
        utterance_span = dur

    return {
        "whisperx_word_count": int(word_count),
        "whisperx_syllable_count": int(syllable_count),
        "whisperx_speech_duration": round(float(speech_duration), 6),
        "whisperx_pause_duration": round(float(pause_duration), 6),
        "whisperx_utterance_span": round(float(utterance_span), 6),
        "whisperx_syllable_rate": round(float(syllable_count / utterance_span), 6),
        "whisperx_word_rate": round(float(word_count / utterance_span), 6),
    }


def classify_global_quartiles(series: pd.Series) -> Tuple[pd.Series, Dict[str, float]]:
    """Classify positive numeric values into global (gender-agnostic) 25/50/25 quartiles (low/moderate/high)."""
    numeric = pd.to_numeric(series, errors="coerce")
    valid = numeric.dropna()
    valid = valid[valid > 0.0]
    if valid.empty:
        raise ValueError("Cannot fit global quartiles: no positive finite values found.")

    q25 = float(valid.quantile(0.25))
    q50 = float(valid.quantile(0.50))
    q75 = float(valid.quantile(0.75))

    def _label(val: Any) -> Optional[str]:
        if val is None or pd.isna(val):
            return None
        v = float(val)
        if not math.isfinite(v) or v <= 0.0:
            return None
        if v < q25:
            return "low"
        if v > q75:
            return "high"
        return "moderate"

    return numeric.map(_label), {"q25": q25, "q50": q50, "q75": q75, "count": int(len(valid))}


def _resolve_clean_train_wav(raw_path: Any, clean_dir: Path = CLEAN_DATA_DIR) -> Path:
    """Locate the local 16 kHz cleaned training WAV without mutating stored CSV path strings."""
    p = Path(str(raw_path))
    if p.is_file():
        return p
    rebased = Path(clean_dir) / "train" / p.name
    if rebased.is_file():
        return rebased
    raise FileNotFoundError(f"Cleaned training WAV not found locally: {raw_path} (also checked {rebased})")


def _load_wav_16k_mono(path: Path, target_sr: int = EMO_TAG_SR) -> Tuple[np.ndarray, float]:
    wav, sr = sf.read(str(path), dtype="float32", always_2d=True)
    wav = wav.mean(axis=1) if wav.shape[1] > 1 else wav[:, 0]
    if sr != target_sr:
        wav = _resample_audio(wav, sr, target_sr)
    return wav.astype(np.float32, copy=False), float(len(wav) / float(target_sr))


def _transcribe_whisperx_batch(
    asr_model: Any,
    batch_wavs: Sequence[np.ndarray],
    batch_size: int = WHISPERX_RATE_BATCH_SIZE,
    language: str = "en",
) -> List[str]:
    """Transcribe up to `batch_size` independent audio clips in a single batched WhisperX GPU pass."""
    if not batch_wavs:
        return []
    try:
        # Reset any interleaved-context state so independent utterances never leak tokens across batch slots
        if hasattr(asr_model, "_use_interleaved_context"):
            asr_model._use_interleaved_context = False
        if hasattr(asr_model, "previous_batch_context_tokens"):
            asr_model.previous_batch_context_tokens = [[]] * len(batch_wavs)

        batch_inputs = [{"inputs": w} for w in batch_wavs]
        raw_outputs = list(asr_model(batch_inputs, batch_size=min(batch_size, len(batch_wavs))))
        if len(raw_outputs) == len(batch_wavs):
            return [
                str(out.get("text", "")).strip() if isinstance(out, dict) else str(out).strip()
                for out in raw_outputs
            ]
    except Exception:
        pass

    # Safe fallback across WhisperX API revisions
    texts: List[str] = []
    for wav in batch_wavs:
        res = asr_model.transcribe(wav, batch_size=batch_size, language=language)
        segs = res.get("segments", []) if isinstance(res, dict) else []
        texts.append(" ".join(str(s.get("text", "")).strip() for s in segs).strip())
    return texts


def _align_whisperx_batch(
    batch_wavs: Sequence[np.ndarray],
    batch_durs: Sequence[float],
    batch_texts: Sequence[str],
    batch_ref_texts: Sequence[str],
    align_model: Any,
    align_metadata: Dict[str, Any],
    device: str,
    align_fn: Any = None,
    sr: int = EMO_TAG_SR,
    pad_sec: float = WHISPERX_RATE_SILENCE_PAD_SEC,
) -> List[List[Dict[str, Any]]]:
    """
    Align a batch of up to 16 utterances with WhisperX forced alignment and return
    per-utterance word timestamps relative to each utterance's own [0, duration] frame.
    """
    if align_fn is None:
        import whisperx
        align_fn = whisperx.align

    n_items = len(batch_wavs)
    results: List[List[Dict[str, Any]]] = [[] for _ in range(n_items)]
    pad_samples = int(round(pad_sec * sr))
    pad_zeros = np.zeros(pad_samples, dtype=np.float32)

    pieces: List[np.ndarray] = []
    batch_segments: List[Dict[str, Any]] = []
    active_map: List[Tuple[int, float, float]] = []
    cursor_samples = 0

    for i, (wav, dur, asr_txt, ref_txt) in enumerate(zip(batch_wavs, batch_durs, batch_texts, batch_ref_texts)):
        txt = asr_txt.strip() if isinstance(asr_txt, str) and asr_txt.strip() else str(ref_txt or "").strip()
        start_sec = cursor_samples / float(sr)
        end_sec = start_sec + dur
        pieces.append(wav)
        if i < n_items - 1:
            pieces.append(pad_zeros)
        cursor_samples += len(wav) + (pad_samples if i < n_items - 1 else 0)
        if txt:
            batch_segments.append({"text": txt, "start": start_sec, "end": end_sec})
            active_map.append((i, start_sec, dur))

    batch_ok = False
    if batch_segments:
        combined_wav = np.concatenate(pieces, axis=0)
        try:
            aligned = align_fn(
                batch_segments,
                align_model,
                align_metadata,
                combined_wav,
                device,
                return_char_alignments=False,
            )
            aligned_segs = aligned.get("segments", []) if isinstance(aligned, dict) else []
            if len(aligned_segs) == len(active_map):
                for seg, (item_idx, offset_sec, dur) in zip(aligned_segs, active_map):
                    shifted: List[Dict[str, Any]] = []
                    for w in seg.get("words", []) or []:
                        if not isinstance(w, dict) or w.get("start") is None or w.get("end") is None:
                            continue
                        s_rel = max(0.0, min(dur, float(w["start"]) - offset_sec))
                        e_rel = max(0.0, min(dur, float(w["end"]) - offset_sec))
                        if e_rel > s_rel:
                            shifted.append({"word": w.get("word", ""), "start": s_rel, "end": e_rel})
                    results[item_idx] = shifted
                batch_ok = True
        except Exception:
            batch_ok = False

    # Per-item fallback if batched alignment failed or any single clip produced 0 timestamped words
    for i, (wav, dur, asr_txt, ref_txt) in enumerate(zip(batch_wavs, batch_durs, batch_texts, batch_ref_texts)):
        if batch_ok and results[i]:
            continue
        candidates = []
        if isinstance(asr_txt, str) and asr_txt.strip():
            candidates.append(asr_txt.strip())
        if isinstance(ref_txt, str) and ref_txt.strip() and ref_txt.strip() not in candidates:
            candidates.append(ref_txt.strip())
        for cand_txt in candidates:
            try:
                single = align_fn(
                    [{"text": cand_txt, "start": 0.0, "end": dur}],
                    align_model,
                    align_metadata,
                    wav,
                    device,
                    return_char_alignments=False,
                )
                words = [
                    {
                        "word": w.get("word", ""),
                        "start": max(0.0, min(dur, float(w["start"]))),
                        "end": max(0.0, min(dur, float(w["end"]))),
                    }
                    for w in (single.get("word_segments", []) if isinstance(single, dict) else [])
                    if isinstance(w, dict) and w.get("start") is not None and w.get("end") is not None
                    and float(w["end"]) > float(w["start"])
                ]
                if words:
                    results[i] = words
                    break
            except Exception:
                continue

    return results


def _atomic_write_csv(df: pd.DataFrame, dst_path: Path) -> None:
    dst_path = Path(dst_path)
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = dst_path.with_suffix(".csv.partial")
    try:
        df.to_csv(tmp_path, index=False)
        tmp_path.replace(dst_path)
    finally:
        tmp_path.unlink(missing_ok=True)


def add_whisperx_speech_rates_to_clean_train(
    train_csv_path: Path = CLEAN_DATA_DIR / "clean_train_set_with_cues.csv",
    drive_dir: Optional[Path] = DRIVE_DATA_DIR,
    batch_size: int = WHISPERX_RATE_BATCH_SIZE,
    overwrite: bool = OVERWRITE_WHISPERX_RATE,
    whisper_model_name: str = "large-v3",
    asr_model: Any = None,
    align_model: Any = None,
    align_metadata: Optional[Dict[str, Any]] = None,
    device: Optional[str] = None,
    align_fn: Any = None,
    syllable_counter: Optional[PhoneticSyllableCounter] = None,
) -> pd.DataFrame:
    """
    Compute WhisperX word-timestamp + pause-duration syllable/word rates (via CMUdict + g2p_en)
    for the full cleaned training set in batches of 16 audios, append global 25/50/25 quartile labels
    (low/moderate/high) into clean_train_set_with_cues.csv, and atomically sync ONLY that CSV to Drive.
    """
    train_csv_path = Path(train_csv_path)
    if not train_csv_path.is_file():
        raise FileNotFoundError(f"Missing cleaned training CSV: {train_csv_path}")

    df = pd.read_csv(train_csv_path)
    if df.empty:
        raise ValueError(f"Cleaned training CSV is empty: {train_csv_path}")

    has_all_cols = all(col in df.columns for col in WHISPERX_RATE_COLUMNS)
    if has_all_cols and not overwrite:
        complete_rates = (
            pd.to_numeric(df["whisperx_syllable_rate"], errors="coerce").notna().all()
            and pd.to_numeric(df["whisperx_word_rate"], errors="coerce").notna().all()
            and df["whisperx_syllable_rate_cue"].isin(["low", "moderate", "high"]).all()
            and df["whisperx_word_rate_cue"].isin(["low", "moderate", "high"]).all()
        )
        if complete_rates:
            print(
                f"[WhisperX Speech Rate] Skipping — all {len(df):,} rows already have complete "
                f"syllable/word rate tracking in {train_csv_path}"
            )
            return df

    clean_dir = train_csv_path.parent
    partial_ckpt = clean_dir / "whisperx_rate_train.partial.csv"
    numeric_cols = (
        "whisperx_word_count",
        "whisperx_syllable_count",
        "whisperx_speech_duration",
        "whisperx_pause_duration",
        "whisperx_utterance_span",
        "whisperx_syllable_rate",
        "whisperx_word_rate",
    )

    # Restore partial progress if a previous Colab session was interrupted mid-run
    done_by_key: Dict[Tuple[int, int], Dict[str, Any]] = {}
    if partial_ckpt.is_file() and not overwrite:
        try:
            part_df = pd.read_csv(partial_ckpt)
            for _, prow in part_df.iterrows():
                if pd.notna(prow.get("whisperx_syllable_rate")) and pd.notna(prow.get("whisperx_word_rate")):
                    key = (int(prow["dialogue"]), int(prow["utterance"]))
                    done_by_key[key] = {col: prow[col] for col in numeric_cols if col in prow}
            if done_by_key:
                print(f"[WhisperX Speech Rate] Resuming from partial checkpoint ({len(done_by_key):,}/{len(df):,} done)")
        except Exception:
            done_by_key = {}

    # Pre-verify local WAV availability on spot-check rows before loading GPU models
    spot_indices = sorted(set(list(range(min(5, len(df)))) + list(range(max(0, len(df) - 5), len(df)))))
    for idx in spot_indices:
        _resolve_clean_train_wav(df.at[idx, "audio_path"], clean_dir=clean_dir)

    pending_indices: List[int] = []
    results_map: Dict[int, Dict[str, Any]] = {}
    for idx, row in df.iterrows():
        key = (int(row["dialogue"]), int(row["utterance"]))
        if key in done_by_key:
            results_map[idx] = done_by_key[key]
        else:
            pending_indices.append(idx)

    # Lazy-load WhisperX ASR + Alignment models and CMUdict/g2p_en only if rows remain pending
    loaded_locally = False
    if pending_indices:
        if syllable_counter is None:
            syllable_counter = PhoneticSyllableCounter()
            # Pre-load CMUdict once before the batch loop
            syllable_counter._ensure_lexicon()

        if asr_model is None or align_model is None or align_metadata is None:
            # Reuse models already held by emo_tag_cleaner if present
            cleaner = globals().get("emo_tag_cleaner")
            if (
                cleaner is not None
                and getattr(cleaner, "asr_model", None) is not None
                and getattr(cleaner, "align_model", None) is not None
                and getattr(cleaner, "align_metadata", None) is not None
            ):
                asr_model = cleaner.asr_model
                align_model = cleaner.align_model
                align_metadata = cleaner.align_metadata
                device = device or cleaner.device
            else:
                import torch
                import whisperx

                device = device or ("cuda" if torch.cuda.is_available() else "cpu")
                compute_type = "float16" if device == "cuda" else "int8"
                print(
                    f"[WhisperX Speech Rate] Loading WhisperX ({whisper_model_name}) + alignment model "
                    f"on {device} ({compute_type})..."
                )
                asr_model = whisperx.load_model(
                    whisper_model_name,
                    device=device,
                    compute_type=compute_type,
                    language="en",
                )
                align_model, align_metadata = whisperx.load_align_model(
                    language_code="en",
                    device=device,
                )
                # One-time silent warmup ensures internal FasterWhisperPipeline tokenizer/context state is initialized
                try:
                    asr_model.transcribe(np.zeros(EMO_TAG_SR, dtype=np.float32), batch_size=batch_size, language="en")
                except Exception:
                    pass
                loaded_locally = True

        total_batches = math.ceil(len(pending_indices) / float(batch_size))
        with tqdm(
            total=len(pending_indices),
            desc=f"WhisperX word/syllable rate (batch={batch_size})",
            unit="audio",
            bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} audios [{elapsed} elapsed, {remaining} remaining, {rate_fmt}]",
        ) as progress:
            for batch_num, start in enumerate(range(0, len(pending_indices), batch_size), start=1):
                batch_idx = pending_indices[start : start + batch_size]
                batch_paths = [_resolve_clean_train_wav(df.at[i, "audio_path"], clean_dir=clean_dir) for i in batch_idx]
                batch_ref_texts = [str(df.at[i, "text"]) if pd.notna(df.at[i, "text"]) else "" for i in batch_idx]

                # Parallel disk I/O for the 16 WAVs in this batch
                with ThreadPoolExecutor(max_workers=min(MAX_WORKERS, len(batch_paths))) as pool:
                    loaded_audio = list(pool.map(_load_wav_16k_mono, batch_paths))
                batch_wavs = [item[0] for item in loaded_audio]
                batch_durs = [item[1] for item in loaded_audio]

                # 1. Batched WhisperX ASR generation (16 audios per GPU pass)
                batch_asr_texts = _transcribe_whisperx_batch(
                    asr_model, batch_wavs, batch_size=batch_size, language="en"
                )

                # 2. Batched WhisperX forced alignment -> word-level timestamps
                batch_words = _align_whisperx_batch(
                    batch_wavs=batch_wavs,
                    batch_durs=batch_durs,
                    batch_texts=batch_asr_texts,
                    batch_ref_texts=batch_ref_texts,
                    align_model=align_model,
                    align_metadata=align_metadata,
                    device=device or "cpu",
                    align_fn=align_fn,
                )

                # 3. Compute CMUdict+g2p_en syllable/word counts, active speech duration, pause duration, and rates
                for idx, words, dur, ref_txt in zip(batch_idx, batch_words, batch_durs, batch_ref_texts):
                    metrics = compute_speech_rate_metrics_from_words(
                        words,
                        audio_duration=dur,
                        syllable_counter=syllable_counter,
                        reference_text=ref_txt,
                    )
                    results_map[idx] = metrics

                progress.set_postfix(batch=f"{batch_num}/{total_batches}")
                progress.update(len(batch_idx))

                # Periodic resumable checkpoint
                if WHISPERX_RATE_CHECKPOINT_EVERY_BATCHES > 0 and (batch_num % WHISPERX_RATE_CHECKPOINT_EVERY_BATCHES == 0):
                    ckpt_rows = [
                        {
                            "dialogue": int(df.at[i, "dialogue"]),
                            "utterance": int(df.at[i, "utterance"]),
                            **results_map[i],
                        }
                        for i in sorted(results_map)
                    ]
                    _atomic_write_csv(pd.DataFrame(ckpt_rows), partial_ckpt)

        total_unique_lookups = syllable_counter.cmu_hits + syllable_counter.g2p_hits
        if total_unique_lookups > 0:
            cmu_pct = 100.0 * syllable_counter.cmu_hits / total_unique_lookups
            print(
                f"[Phonetic Lexicon] Unique word types resolved -> "
                f"CMUdict: {syllable_counter.cmu_hits:,} ({cmu_pct:.2f}%), "
                f"g2p_en OOV fallback: {syllable_counter.g2p_hits:,} ({100.0 - cmu_pct:.2f}%)"
            )

        if loaded_locally:
            del asr_model, align_model
            gc.collect()
            try:
                import torch
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            except Exception:
                pass

    # Attach numeric WhisperX tracking columns without altering any pre-existing columns
    for col in numeric_cols:
        df[col] = [results_map[i][col] for i in range(len(df))]

    # Global (gender-agnostic) 25% low / 50% moderate / 25% high quartile classification
    df["whisperx_syllable_rate_cue"], syl_q = classify_global_quartiles(df["whisperx_syllable_rate"])
    df["whisperx_word_rate_cue"], wrd_q = classify_global_quartiles(df["whisperx_word_rate"])

    # Save updated clean_train_set_with_cues.csv atomically locally
    _atomic_write_csv(df, train_csv_path)
    if partial_ckpt.exists():
        partial_ckpt.unlink()

    # Atomically sync ONLY the updated clean_train_set_with_cues.csv to Google Drive
    if drive_dir is not None:
        drive_train_csv = Path(drive_dir) / "emo_tag_clean" / train_csv_path.name
        _atomic_write_csv(df, drive_train_csv)
        print(f"[WhisperX Speech Rate] Synced updated CSV to Drive -> {drive_train_csv}")

    print(f"\n=== Global WhisperX True Speaking Rate Quartiles (N={len(df):,}, Gender-Agnostic 25/50/25) ===")
    print(
        f"  Syllable rate (syl/s) -> Q25={syl_q['q25']:.4f}, Q50={syl_q['q50']:.4f}, Q75={syl_q['q75']:.4f} "
        f"(n={syl_q['count']:,})"
    )
    print(
        f"  Word rate (words/s)   -> Q25={wrd_q['q25']:.4f}, Q50={wrd_q['q50']:.4f}, Q75={wrd_q['q75']:.4f} "
        f"(n={wrd_q['count']:,})"
    )
    for cue_col in ("whisperx_syllable_rate_cue", "whisperx_word_rate_cue"):
        counts = df[cue_col].value_counts(dropna=False)
        pcts = (df[cue_col].value_counts(normalize=True, dropna=False) * 100.0).round(2)
        summary = ", ".join(f"{k}: {counts[k]:,} ({pcts[k]}%)" for k in counts.index)
        print(f"  - {cue_col:<27}: {summary}")

    print(f"Saved updated cleaned training CSV -> {train_csv_path} ({len(df):,} rows, {len(df.columns)} columns)")
    return df


clean_train_set = add_whisperx_speech_rates_to_clean_train(
    train_csv_path=CLEAN_DATA_DIR / "clean_train_set_with_cues.csv",
    drive_dir=DRIVE_DATA_DIR,
    batch_size=WHISPERX_RATE_BATCH_SIZE,
    overwrite=OVERWRITE_WHISPERX_RATE,
)

[WhisperX Speech Rate] Resuming from partial checkpoint (3,600/6,059 done)
[WhisperX Speech Rate] Loading WhisperX (large-v3) + alignment model on cuda (float16)...
2026-11-24 20:56:37 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


INFO: Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.6. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.13/dist-packages/whisperx/assets/pytorch_model.bin`
INFO:lightning.pytorch.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.6. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.13/dist-packages/whisperx/assets/pytorch_model.bin`


2026-11-24 20:56:38 - whisperx.vads.pyannote - WARNING - No active speech found in audio


WhisperX word/syllable rate (batch=32):   0%|          | 0/2459 audios [00:00 elapsed, ? remaining, ?audio/s]

[Phonetic Lexicon] Unique word types resolved -> CMUdict: 2,418 (96.07%), g2p_en OOV fallback: 99 (3.93%)
[WhisperX Speech Rate] Synced updated CSV to Drive -> /content/drive/MyDrive/AGORA_SER/data/meld_16k_mono/emo_tag_clean/clean_train_set_with_cues.csv

=== Global WhisperX True Speaking Rate Quartiles (N=6,059, Gender-Agnostic 25/50/25) ===
  Syllable rate (syl/s) -> Q25=4.4643, Q50=5.5556, Q75=6.7021 (n=6,059)
  Word rate (words/s)   -> Q25=3.4706, Q50=4.4477, Q75=5.4828 (n=6,059)
  - whisperx_syllable_rate_cue : moderate: 3,032 (50.04%), high: 1,515 (25.0%), low: 1,512 (24.95%)
  - whisperx_word_rate_cue     : moderate: 3,029 (49.99%), low: 1,515 (25.0%), high: 1,515 (25.0%)
Saved updated cleaned training CSV -> /content/agora_work/meld_16k_mono/emo_tag_clean/clean_train_set_with_cues.csv (6,059 rows, 40 columns)


## 11. Batched WhisperX True Syllable & Word Rate Tracking (Cleaned Validation Set)
Runs the same batched **WhisperX (`large-v3` + wav2vec2 forced alignment, 16 audios per batch)** and **CMUdict + `g2p_en`** phonetic syllable counter over the cleaned validation set (`clean_val_set_with_cues.csv`), then assigns global 25/50/25 quartile labels (`low` / `moderate` / `high`) using **strictly the training-fitted global `Q25`/`Q75` thresholds** from `clean_train_set_with_cues.csv` (zero validation leakage):
- Appends the exact same 9 columns (`whisperx_word_count`, `whisperx_syllable_count`, `whisperx_speech_duration`, `whisperx_pause_duration`, `whisperx_utterance_span`, `whisperx_syllable_rate`, `whisperx_syllable_rate_cue`, `whisperx_word_rate`, `whisperx_word_rate_cue`) into `clean_val_set_with_cues.csv` while preserving all existing openSMILE columns intact.
- **Does not overwrite** `clean_train_set_with_cues.csv`, `gender_acoustic_quartiles.json`, existing audio WAVs, or `meld_16k_mono_clean.tar`.
- **Idempotent on Run All**: skips immediately with zero WhisperX inference if `clean_val_set_with_cues.csv` already contains complete WhisperX speech-rate columns (set `OVERWRITE_WHISPERX_RATE_VAL = True` to recompute).
OVERWRITE_WHISPERX_RATE_VAL = False

In [22]:
OVERWRITE_WHISPERX_RATE_VAL = False

def fit_global_quartile_thresholds(series: pd.Series) -> Dict[str, float]:
    """Fit global (gender-agnostic) Q25/Q50/Q75 thresholds strictly on positive finite training values."""
    numeric = pd.to_numeric(series, errors="coerce")
    valid = numeric.dropna()
    valid = valid[valid > 0.0]
    if valid.empty:
        raise ValueError("Cannot fit global training quartile thresholds: no positive finite values found.")
    return {
        "q25": float(valid.quantile(0.25)),
        "q50": float(valid.quantile(0.50)),
        "q75": float(valid.quantile(0.75)),
        "count": int(len(valid)),
    }


def apply_global_quartiles(series: pd.Series, thresholds: Dict[str, float]) -> pd.Series:
    """Classify positive numeric values into low/moderate/high using pre-fitted training Q25/Q75 cutoffs."""
    q25 = float(thresholds["q25"])
    q75 = float(thresholds["q75"])
    numeric = pd.to_numeric(series, errors="coerce")

    def _label(val: Any) -> Optional[str]:
        if val is None or pd.isna(val):
            return None
        v = float(val)
        if not math.isfinite(v) or v <= 0.0:
            return None
        if v < q25:
            return "low"
        if v > q75:
            return "high"
        return "moderate"

    return numeric.map(_label)


def _resolve_clean_val_wav(raw_path: Any, clean_dir: Path = CLEAN_DATA_DIR) -> Path:
    """Locate the local 16 kHz cleaned validation WAV without mutating stored CSV path strings."""
    p = Path(str(raw_path))
    if p.is_file():
        return p
    rebased = Path(clean_dir) / "val" / p.name
    if rebased.is_file():
        return rebased
    raise FileNotFoundError(f"Cleaned validation WAV not found locally: {raw_path} (also checked {rebased})")


def _val_whisperx_columns_complete(val_df: pd.DataFrame) -> bool:
    """Return True when all validation rows have already been scored and labeled."""
    if not all(col in val_df.columns for col in WHISPERX_RATE_COLUMNS):
        return False
    has_counts = (
        pd.to_numeric(val_df["whisperx_word_count"], errors="coerce").notna().all()
        and pd.to_numeric(val_df["whisperx_syllable_count"], errors="coerce").notna().all()
    )
    if not has_counts:
        return False
    syl_num = pd.to_numeric(val_df["whisperx_syllable_rate"], errors="coerce")
    wrd_num = pd.to_numeric(val_df["whisperx_word_rate"], errors="coerce")
    valid_syl = syl_num.notna() & (syl_num > 0.0)
    valid_wrd = wrd_num.notna() & (wrd_num > 0.0)
    if not (valid_syl.any() and valid_wrd.any()):
        return False
    return bool(
        val_df.loc[valid_syl, "whisperx_syllable_rate_cue"].isin(["low", "moderate", "high"]).all()
        and val_df.loc[valid_wrd, "whisperx_word_rate_cue"].isin(["low", "moderate", "high"]).all()
    )


def add_whisperx_speech_rates_to_clean_val(
    val_csv_path: Path = CLEAN_DATA_DIR / "clean_val_set_with_cues.csv",
    train_csv_path: Path = CLEAN_DATA_DIR / "clean_train_set_with_cues.csv",
    drive_dir: Optional[Path] = DRIVE_DATA_DIR,
    batch_size: int = WHISPERX_RATE_BATCH_SIZE,
    overwrite: bool = OVERWRITE_WHISPERX_RATE_VAL,
    whisper_model_name: str = "large-v3",
    asr_model: Any = None,
    align_model: Any = None,
    align_metadata: Optional[Dict[str, Any]] = None,
    device: Optional[str] = None,
    align_fn: Any = None,
    syllable_counter: Optional[PhoneticSyllableCounter] = None,
) -> pd.DataFrame:
    """
    Compute WhisperX word-timestamp + pause-duration syllable/word rates (CMUdict + g2p_en)
    for the cleaned validation set in batches of 16 audios, classify using strictly the
    training-fitted global Q25/Q75 cutoffs (zero leakage), and atomically sync ONLY
    clean_val_set_with_cues.csv to Drive.
    """
    val_csv_path = Path(val_csv_path)
    train_csv_path = Path(train_csv_path)
    if not val_csv_path.is_file():
        raise FileNotFoundError(f"Missing cleaned validation CSV: {val_csv_path}")
    if not train_csv_path.is_file():
        raise FileNotFoundError(f"Missing cleaned training CSV (needed for zero-leakage global cutoffs): {train_csv_path}")

    val_df = pd.read_csv(val_csv_path)
    if val_df.empty:
        raise ValueError(f"Cleaned validation CSV is empty: {val_csv_path}")

    if not overwrite and _val_whisperx_columns_complete(val_df):
        print(
            f"[WhisperX Speech Rate — Val] Skipping — all {len(val_df):,} rows already have complete "
            f"syllable/word rate tracking in {val_csv_path}"
        )
        return val_df

    train_df = pd.read_csv(train_csv_path)
    for req_col in ("whisperx_syllable_rate", "whisperx_word_rate"):
        if req_col not in train_df.columns:
            raise ValueError(
                f"Training CSV {train_csv_path} is missing '{req_col}'; run Section 10 on train first."
            )
    syl_q = fit_global_quartile_thresholds(train_df["whisperx_syllable_rate"])
    wrd_q = fit_global_quartile_thresholds(train_df["whisperx_word_rate"])

    clean_dir = val_csv_path.parent
    partial_ckpt = clean_dir / "whisperx_rate_val.partial.csv"
    numeric_cols = (
        "whisperx_word_count",
        "whisperx_syllable_count",
        "whisperx_speech_duration",
        "whisperx_pause_duration",
        "whisperx_utterance_span",
        "whisperx_syllable_rate",
        "whisperx_word_rate",
    )

    done_by_key: Dict[Tuple[int, int], Dict[str, Any]] = {}
    if partial_ckpt.is_file() and not overwrite:
        try:
            part_df = pd.read_csv(partial_ckpt)
            for _, prow in part_df.iterrows():
                if pd.notna(prow.get("whisperx_word_count")):
                    key = (int(prow["dialogue"]), int(prow["utterance"]))
                    done_by_key[key] = {col: prow[col] for col in numeric_cols if col in prow}
            if done_by_key:
                print(
                    f"[WhisperX Speech Rate — Val] Resuming from partial checkpoint "
                    f"({len(done_by_key):,}/{len(val_df):,} done)"
                )
        except Exception:
            done_by_key = {}

    spot_indices = sorted(set(list(range(min(5, len(val_df)))) + list(range(max(0, len(val_df) - 5), len(val_df)))))
    for idx in spot_indices:
        _resolve_clean_val_wav(val_df.at[idx, "audio_path"], clean_dir=clean_dir)

    pending_indices: List[int] = []
    results_map: Dict[int, Dict[str, Any]] = {}
    for idx, row in val_df.iterrows():
        key = (int(row["dialogue"]), int(row["utterance"]))
        if key in done_by_key:
            results_map[idx] = done_by_key[key]
        else:
            pending_indices.append(idx)

    loaded_locally = False
    if pending_indices:
        if syllable_counter is None:
            syllable_counter = PhoneticSyllableCounter()
            syllable_counter._ensure_lexicon()

        if asr_model is None or align_model is None or align_metadata is None:
            cleaner = globals().get("emo_tag_cleaner")
            if (
                cleaner is not None
                and getattr(cleaner, "asr_model", None) is not None
                and getattr(cleaner, "align_model", None) is not None
                and getattr(cleaner, "align_metadata", None) is not None
            ):
                asr_model = cleaner.asr_model
                align_model = cleaner.align_model
                align_metadata = cleaner.align_metadata
                device = device or cleaner.device
            else:
                import torch
                import whisperx

                device = device or ("cuda" if torch.cuda.is_available() else "cpu")
                compute_type = "float16" if device == "cuda" else "int8"
                print(
                    f"[WhisperX Speech Rate — Val] Loading WhisperX ({whisper_model_name}) + alignment model "
                    f"on {device} ({compute_type})..."
                )
                asr_model = whisperx.load_model(
                    whisper_model_name,
                    device=device,
                    compute_type=compute_type,
                    language="en",
                )
                align_model, align_metadata = whisperx.load_align_model(
                    language_code="en",
                    device=device,
                )
                try:
                    asr_model.transcribe(np.zeros(EMO_TAG_SR, dtype=np.float32), batch_size=batch_size, language="en")
                except Exception:
                    pass
                loaded_locally = True

        total_batches = math.ceil(len(pending_indices) / float(batch_size))
        with tqdm(
            total=len(pending_indices),
            desc=f"WhisperX val word/syllable rate (batch={batch_size})",
            unit="audio",
            bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} audios [{elapsed} elapsed, {remaining} remaining, {rate_fmt}]",
        ) as progress:
            for batch_num, start in enumerate(range(0, len(pending_indices), batch_size), start=1):
                batch_idx = pending_indices[start : start + batch_size]
                batch_paths = [_resolve_clean_val_wav(val_df.at[i, "audio_path"], clean_dir=clean_dir) for i in batch_idx]
                batch_ref_texts = [str(val_df.at[i, "text"]) if pd.notna(val_df.at[i, "text"]) else "" for i in batch_idx]

                with ThreadPoolExecutor(max_workers=min(MAX_WORKERS, len(batch_paths))) as pool:
                    loaded_audio = list(pool.map(_load_wav_16k_mono, batch_paths))
                batch_wavs = [item[0] for item in loaded_audio]
                batch_durs = [item[1] for item in loaded_audio]

                batch_asr_texts = _transcribe_whisperx_batch(
                    asr_model, batch_wavs, batch_size=batch_size, language="en"
                )
                batch_words = _align_whisperx_batch(
                    batch_wavs=batch_wavs,
                    batch_durs=batch_durs,
                    batch_texts=batch_asr_texts,
                    batch_ref_texts=batch_ref_texts,
                    align_model=align_model,
                    align_metadata=align_metadata,
                    device=device or "cpu",
                    align_fn=align_fn,
                )

                for idx, words, dur, ref_txt in zip(batch_idx, batch_words, batch_durs, batch_ref_texts):
                    results_map[idx] = compute_speech_rate_metrics_from_words(
                        words,
                        audio_duration=dur,
                        syllable_counter=syllable_counter,
                        reference_text=ref_txt,
                    )

                progress.set_postfix(batch=f"{batch_num}/{total_batches}")
                progress.update(len(batch_idx))

                if WHISPERX_RATE_CHECKPOINT_EVERY_BATCHES > 0 and (batch_num % WHISPERX_RATE_CHECKPOINT_EVERY_BATCHES == 0):
                    ckpt_rows = [
                        {
                            "dialogue": int(val_df.at[i, "dialogue"]),
                            "utterance": int(val_df.at[i, "utterance"]),
                            **results_map[i],
                        }
                        for i in sorted(results_map)
                    ]
                    _atomic_write_csv(pd.DataFrame(ckpt_rows), partial_ckpt)

        total_unique_lookups = syllable_counter.cmu_hits + syllable_counter.g2p_hits
        if total_unique_lookups > 0:
            cmu_pct = 100.0 * syllable_counter.cmu_hits / total_unique_lookups
            print(
                f"[Phonetic Lexicon — Val] Unique word types resolved -> "
                f"CMUdict: {syllable_counter.cmu_hits:,} ({cmu_pct:.2f}%), "
                f"g2p_en OOV fallback: {syllable_counter.g2p_hits:,} ({100.0 - cmu_pct:.2f}%)"
            )

        if loaded_locally:
            del asr_model, align_model
            gc.collect()
            try:
                import torch
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            except Exception:
                pass

    for col in numeric_cols:
        val_df[col] = [results_map[i][col] for i in range(len(val_df))]

    # Apply strictly the training-fitted global Q25/Q75 cutoffs (zero validation leakage)
    val_df["whisperx_syllable_rate_cue"] = apply_global_quartiles(val_df["whisperx_syllable_rate"], syl_q)
    val_df["whisperx_word_rate_cue"] = apply_global_quartiles(val_df["whisperx_word_rate"], wrd_q)

    _atomic_write_csv(val_df, val_csv_path)
    if partial_ckpt.exists():
        partial_ckpt.unlink()

    if drive_dir is not None:
        drive_val_csv = Path(drive_dir) / "emo_tag_clean" / val_csv_path.name
        _atomic_write_csv(val_df, drive_val_csv)
        print(f"[WhisperX Speech Rate — Val] Synced updated CSV to Drive -> {drive_val_csv}")

    print(
        f"\n=== Validation WhisperX Speaking Rate Labels (N={len(val_df):,}, "
        f"Train-Fitted Global Cutoffs) ==="
    )
    print(
        f"  Train Syllable rate cutoffs -> Q25={syl_q['q25']:.4f}, Q50={syl_q['q50']:.4f}, Q75={syl_q['q75']:.4f} "
        f"(train n={syl_q['count']:,})"
    )
    print(
        f"  Train Word rate cutoffs     -> Q25={wrd_q['q25']:.4f}, Q50={wrd_q['q50']:.4f}, Q75={wrd_q['q75']:.4f} "
        f"(train n={wrd_q['count']:,})"
    )
    for cue_col in ("whisperx_syllable_rate_cue", "whisperx_word_rate_cue"):
        counts = val_df[cue_col].value_counts(dropna=False)
        pcts = (val_df[cue_col].value_counts(normalize=True, dropna=False) * 100.0).round(2)
        summary = ", ".join(f"{k}: {counts[k]:,} ({pcts[k]}%)" for k in counts.index)
        print(f"  - {cue_col:<27}: {summary}")

    print(f"Saved updated cleaned validation CSV -> {val_csv_path} ({len(val_df):,} rows, {len(val_df.columns)} columns)")
    return val_df


clean_val_set = add_whisperx_speech_rates_to_clean_val(
    val_csv_path=CLEAN_DATA_DIR / "clean_val_set_with_cues.csv",
    train_csv_path=CLEAN_DATA_DIR / "clean_train_set_with_cues.csv",
    drive_dir=DRIVE_DATA_DIR,
    batch_size=WHISPERX_RATE_BATCH_SIZE,
    overwrite=OVERWRITE_WHISPERX_RATE_VAL,
)
if IS_COLAB:
    from google.colab import runtime
    runtime.unassign()

[WhisperX Speech Rate — Val] Loading WhisperX (large-v3) + alignment model on cuda (float16)...
2026-11-24 21:10:57 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


INFO: Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.6. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.13/dist-packages/whisperx/assets/pytorch_model.bin`
INFO:lightning.pytorch.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.6. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.13/dist-packages/whisperx/assets/pytorch_model.bin`


2026-11-24 21:10:58 - whisperx.vads.pyannote - WARNING - No active speech found in audio


WhisperX val word/syllable rate (batch=32):   0%|          | 0/816 audios [00:00 elapsed, ? remaining, ?audio/…

2026-11-24 21:12:04 - whisperx.alignment - WARNING - Failed to align segment ("THANKS FOR JOINING US."): backtrack failed, resorting to original
2026-11-24 21:12:05 - whisperx.alignment - WARNING - Failed to align segment ("THANKS FOR JOINING US."): backtrack failed, resorting to original
[Phonetic Lexicon — Val] Unique word types resolved -> CMUdict: 1,174 (97.51%), g2p_en OOV fallback: 30 (2.49%)
[WhisperX Speech Rate — Val] Synced updated CSV to Drive -> /content/drive/MyDrive/AGORA_SER/data/meld_16k_mono/emo_tag_clean/clean_val_set_with_cues.csv

=== Validation WhisperX Speaking Rate Labels (N=816, Train-Fitted Global Cutoffs) ===
  Train Syllable rate cutoffs -> Q25=4.4643, Q50=5.5556, Q75=6.7021 (train n=6,059)
  Train Word rate cutoffs     -> Q25=3.4706, Q50=4.4477, Q75=5.4828 (train n=6,059)
  - whisperx_syllable_rate_cue : moderate: 392 (48.04%), low: 218 (26.72%), high: 206 (25.25%)
  - whisperx_word_rate_cue     : moderate: 417 (51.1%), low: 200 (24.51%), high: 199 (24.39%)


In [ ]:
if IS_COLAB:
    from time import time
    time.sleep(10)
    from google.colab import runtime
    runtime.unassign()